# Pipeline Financeiro 2026 — Versão Final Profissional

Notebook único para preparar, validar e exportar o dataset financeiro de 2026.

## Princípios

- dados `raw` nunca são alterados;
- cada regra de negócio é aplicada uma única vez;
- `CODACT 13` é excluído do universo financeiro;
- KILOSPO é calculado integralmente, mas apenas o que associa ao `inform_27` entra no financeiro;
- diferenças Danone são classificadas, não forçadas;
- Setembro permanece no dataset, mas é excluído da reconciliação Danone enquanto estiver incompleto;
- ocupação é calculada ao nível da viagem (`CODEUT`);
- divisões por zero devolvem `NaN`;
- a exportação só ocorre depois das validações finais.

## 00. Configuração

In [28]:
# ==========================
# 1. Imports
# ==========================
from pathlib import Path
import platform
import duckdb
import numpy as np
import pandas as pd

# ==========================
# 2. Parâmetros
# ==========================
ANO = 2026
MES_VALIDACAO_DANONE_MAX = 10
INGRESSO_CAPILAR_MENSAL = {9: 93_000.0, 10: 85_000.0}

CUSTO_ESTRUTURA_MENSAL = 103_000.0
EXTRA_DANONE_MENSAL = 20_000.0
DESFASAMENTO_MES_COMBUSTIVEL = -1

CODACT_ESPANHA = [338, 211, 15]
CODACT_EXCLUIR = [13]

TOLERANCIA_FINANCEIRA = 0.01

MAPA_CAPACIDADE = {
    4: 6,
    5: 6,
    6: 6,
    8: 12,
    12: 12,
    14: 14,
    15: 16,
    16: 16,
    18: 18,
    20: 20,
    22: 22,
    24: 24,
    33: 33,
    55: 55,
    66: 66,
}

# ==========================
# 3. Caminhos
# ==========================
def get_paths():
    sistema = platform.system()

    if sistema == "Windows":
        base = Path(r"C:\Users\LISARR\OneDrive - Salvesen Logística S.A\python\01.Financeiro")
        return {
            "db": Path(r"C:\Users\LISARR\OneDrive - Salvesen Logística S.A\python\00.DB\2026.duckdb"),
            "kilospo": base / "kilospo" / "INFKILP.CSV",
            "parquet": base / "inform_27_2026_final.parquet",
            "viagens": base / "viagens_2026.parquet",
            "validacao": base / "validacao_financeira_2026.csv",
            "danone_diag": base / "validacao_danone_2026.csv",
        }

    if sistema == "Darwin":
        base = Path(
            "/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen"
        )
        return {
            "db": base / "00_DB" / "2026.duckdb",
            "kilospo": base / "kilospo" / "INFKILP.CSV",
            "parquet": base / "inform_27_2026_final.parquet",
            "viagens": base / "viagens_2026.parquet",
            "validacao": base / "validacao_financeira_2026.csv",
            "danone_diag": base / "validacao_danone_2026.csv",
        }

    base = Path.cwd()
    return {
        "db": base / "2026.duckdb",
        "kilospo": base / "kilospo" / "INFKILP.CSV",
        "parquet": base / "inform_27_2026_final.parquet",
        "viagens": base / "viagens_2026.parquet",
        "validacao": base / "validacao_financeira_2026.csv",
        "danone_diag": base / "validacao_danone_2026.csv",
    }

PATHS = get_paths()


## 01. Funções comuns

In [29]:
# ==========================
# 1. Datas
# ==========================
def normalizar_data(serie):
    texto = (
        serie.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

    data = pd.to_datetime(
        texto,
        format="%Y%m%d",
        errors="coerce",
    )

    mask = data.isna()

    if mask.any():
        data.loc[mask] = pd.to_datetime(
            texto.loc[mask],
            errors="coerce",
        )

    return data

# ==========================
# 2. Referências
# ==========================
def normalizar_referencia(serie):
    return (
        serie.astype("string")
        .str.strip()
        .str.extract(r"(\d{10})", expand=False)
        .astype("string")
    )

# ==========================
# 3. Códigos postais
# ==========================
def normalizar_cp(serie):
    cp = (
        serie.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.replace(r"\D", "", regex=True)
        .str[:7]
    )

    invalido = (
        cp.str.len().fillna(0).lt(4)
        | cp.eq("0").fillna(False)
        | cp.eq("").fillna(False)
    )

    return cp.mask(invalido, pd.NA)

# ==========================
# 4. Divisão segura
# ==========================
def dividir_seguro(numerador, denominador):
    n = pd.to_numeric(numerador, errors="coerce")
    d = pd.to_numeric(denominador, errors="coerce")
    return n.div(d.where(d.ne(0)))

# ==========================
# 5. Moda segura
# ==========================
def moda_segura(serie):
    serie = serie.dropna()

    if serie.empty:
        return np.nan

    moda = serie.mode()

    if not moda.empty:
        return moda.iloc[0]

    return serie.iloc[0]

## 02. Regras de combustível

In [30]:

# ==========================
# 1. Taxas por CODACT e mês
# ==========================
taxas_ingresso = {
    11: {
        3: 0.0637, 4: 0.1043, 5: 0.0869,
        6: 0.0569, 7: 0.0665, 8: 0.1000,
        9: 0.1229,
    },

    74: {
        3: 0.0646, 4: 0.1003, 5: 0.0850,
        6: 0.0585, 7: 0.0670, 8: 0.0965,
        9: 0.1167,
    },

    84: {
        3: 0.0646, 4: 0.1003, 5: 0.0850,
        6: 0.0585, 7: 0.0670, 8: 0.0965,
        9: 0.1167,
    },

    118: {
        1: 0.0849, 2: 0.0974, 3: 0.1860,
        4: 0.2434, 5: 0.2188, 6: 0.1763,
        7: 0.1900, 8: 0.2373, 9: 0.2698,
    },

    131: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },

    262: {
        3: 0.0646, 4: 0.1003, 5: 0.0850,
        6: 0.0585, 7: 0.0670, 8: 0.0965,
        9: 0.1167,
    },

    275: {
        3: 0.0669, 4: 0.1077, 5: 0.0902,
        6: 0.0600, 7: 0.0697, 8: 0.1034,
        9: 0.1265,
    },

    280: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },

    282: {
        3: 0.0646, 4: 0.1003, 5: 0.0850,
        6: 0.0585, 7: 0.0670, 8: 0.0965,
        9: 0.1167,
    },

    332: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },

    391: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },

    398: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },

    403: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },

    404: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },

    405: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },

    415: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },

    418: {
        9: 0.1361,
    },

    419: {
        1: 0.0442, 2: 0.0531, 3: 0.1160,
        4: 0.1568, 5: 0.1393, 6: 0.1091,
        7: 0.1188, 8: 0.1524, 9: 0.1755,
    },

    429: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },

    588: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },

    660: {
        3: 0.0480, 4: 0.0821, 5: 0.0675,
        6: 0.0422, 7: 0.0504, 8: 0.0785,
        9: 0.0978,
    },

    680: {
        3: 0.0646, 4: 0.1003, 5: 0.0850,
        6: 0.0585, 7: 0.0670, 8: 0.0965,
        9: 0.1167,
    },

    784: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },
}


## 03. Importações

In [31]:
# ==========================
# 1. Validar fontes
# ==========================
if not PATHS["db"].exists():
    raise FileNotFoundError(
        f"DuckDB não encontrada: {PATHS['db']}"
    )

if not PATHS["kilospo"].exists():
    raise FileNotFoundError(
        f"INFKILP não encontrado: {PATHS['kilospo']}"
    )

# ==========================
# 2. Ler DuckDB
# ==========================
with duckdb.connect(
    str(PATHS["db"]),
    read_only=True,
) as con:
    df_raw = con.sql("""
        SELECT *
        FROM inform_27_2026
    """).df()

    df_tarifas_raw = con.sql("""
        SELECT
            tipo_local,
            modelo_ingresso,
            tarifa_base,
            data_inicio,
            data_fim
        FROM Danone_Tarifas_2026
    """).df()

    df_clientes_raw = con.sql("""
        SELECT pcodcl, pnomcl, ruta_wh, tipo_local, data_inicio, data_fim
        FROM danone_clientes_2026
    """).df()

    coords_raw = con.sql("""
        SELECT
            cp AS CP,
            point_x AS POINT_X,
            point_y AS POINT_Y
        FROM coordenadas
    """).df()

# ==========================
# 3. Ler KILOSPO CSV
# ==========================
df_kilospo_raw = pd.read_csv(
    PATHS["kilospo"],
    sep=";",
    encoding="cp1252",
    dtype="string",
    keep_default_na=False,
)
df_kilospo_raw.columns = df_kilospo_raw.columns.str.strip()

# ==========================
# 4. Validar importações
# ==========================
pd.DataFrame({
    "fonte": [
        "inform_27_2026",
        "INFKILP.CSV",
        "danone_clientes_2026",
        "Danone_Tarifas_2026",
        "coordenadas",
    ],
    "linhas": [
        len(df_raw),
        len(df_kilospo_raw),
        len(df_clientes_raw),
        len(df_tarifas_raw),
        len(coords_raw),
    ],
})


,fonte,linhas
0,inform_27_2026,817764
1,INFKILP.CSV,16061
2,danone_clientes_2026,1047
3,Danone_Tarifas_2026,37
4,coordenadas,178409


## 04. Universo financeiro — `inform_27_2026`

In [32]:
# ==========================
# 1. Cópia de trabalho
# ==========================
df_base = df_raw.copy()

# ==========================
# 2. Texto
# ==========================
for coluna in df_base.select_dtypes(
    include=["object"]
).columns:
    df_base[coluna] = (
        df_base[coluna]
        .astype("string")
        .str.strip()
    )

# ==========================
# 3. Numéricos
# ==========================
colunas_numericas = [
    "CODACT",
    "INGRESODT",
    "COSTEDT",
    "RENTADT",
    "PALETSDT",
    "PESO_BRUTO",
    "PALETS",
    "KM",
    "KMREALES",
    "CAMION_CAPACIDAD",
]

for coluna in colunas_numericas:
    if coluna in df_base.columns:
        valores = (
            df_base[coluna]
            .astype("string")
            .str.strip()
            .str.replace(",", ".", regex=False)
        )
        df_base[coluna] = pd.to_numeric(
            valores,
            errors="coerce",
        )

# ==========================
# 3.1. Validar financeiros importados
# ==========================
mes_origem = (
    df_base["FENTREGA"]
    .astype("string")
    .str.strip()
    .str[:6]
)

mask_meses_corrigidos = mes_origem.isin(["202609", "202610"])

validacao_importacao = (
    df_base.loc[mask_meses_corrigidos]
    .assign(MES=mes_origem.loc[mask_meses_corrigidos])
    .groupby("MES", as_index=False)
    .agg(
        linhas=("FENTREGA", "size"),
        ingresso_validos=("INGRESODT", "count"),
        custo_validos=("COSTEDT", "count"),
        ingresso=("INGRESODT", "sum"),
        custo=("COSTEDT", "sum"),
    )
)

if (
    validacao_importacao.empty
    or set(validacao_importacao["MES"]) != {"202609", "202610"}
    or not validacao_importacao["linhas"].eq(validacao_importacao["ingresso_validos"]).all()
    or not validacao_importacao["linhas"].eq(validacao_importacao["custo_validos"]).all()
):
    raise ValueError(
        "Os ingressos/custos de Setembro e Outubro não foram totalmente convertidos. "
        "Verifica o formato dos valores na tabela inform_27_2026."
    )

# ==========================
# 4. CODEUT
# ==========================
df_base["CODEUT"] = (
    df_base["CODEUT"]
    .astype("string")
    .str.strip()
    .replace({
        "": pd.NA,
        "nan": pd.NA,
        "None": pd.NA,
        "<NA>": pd.NA,
    })
)

# ==========================
# 5. Datas
# ==========================
df_base["FCARGA"] = normalizar_data(
    df_base["FCARGA"]
)

df_base["FENTREGA"] = normalizar_data(
    df_base["FENTREGA"]
)

# ==========================
# 6. Chave de referência
# ==========================
df_base["referencia_chave"] = (
    normalizar_referencia(
        df_base["REFERENCIA"]
    )
)

# ==========================
# 7. Guardar universo antes dos filtros
# ==========================
df_raw_check = df_base.copy()

# ==========================
# 8. Ajustes de ingresso
# ==========================
df_base["INGRESODT_ORIGINAL"] = (
    df_base["INGRESODT"]
)

df_base.loc[
    df_base["CODACT"].eq(11),
    "INGRESODT",
] = 0.0

mask_espanha = df_base["CODACT"].isin(
    CODACT_ESPANHA
)

df_base.loc[
    mask_espanha,
    "INGRESODT",
] = (
    df_base.loc[
        mask_espanha,
        "COSTEDT",
    ]
    * 1.05
)

# ==========================
# 9. Reservar CODACT 013 para CAPILAR em Setembro e Outubro
# ==========================
mes_origem_capilar = df_base["FENTREGA"].dt.month
mask_013_capilar = (
    df_base["CODACT"].eq(13)
    & df_base["FENTREGA"].dt.year.eq(ANO)
    & mes_origem_capilar.isin(INGRESSO_CAPILAR_MENSAL)
)
df_base = df_base.loc[
    ~df_base["CODACT"].isin(CODACT_EXCLUIR)
    | mask_013_capilar
].copy()

# O ingresso fixo CAPILAR substitui o ingresso original do 013.
df_base.loc[df_base["CODACT"].eq(13), "INGRESODT"] = 0.0

# ==========================
# 10. Filtro operacional
# ==========================
mask_gestao = (
    df_base["GESTION"].eq("LIS")
    | (
        df_base["GESTION"].isna()
        & df_base["PROPIETARIO"].isin(
            ["CEP", "PTG"]
        )
    )
)

df_base = df_base.loc[
    mask_gestao
].copy()

# ==========================
# 11. ID e calendário
# ==========================
df_base["_linha_id"] = np.arange(
    len(df_base),
    dtype="int64",
)

df_base["data"] = df_base["FENTREGA"]
df_base["ano"] = df_base["data"].dt.year
df_base["mes"] = df_base["data"].dt.month
df_base["mes_nome"] = df_base["data"].dt.month_name()
df_base["week_number"] = (
    df_base["data"]
    .dt.isocalendar()
    .week
    .astype("Int64")
)
df_base["week_day"] = (
    df_base["data"]
    .dt.day_name()
)
df_base["mes_estrutura"] = (
    df_base["data"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

# ==========================
# 12. Validação
# ==========================
pd.DataFrame({
    "metrica": [
        "Linhas raw",
        "Linhas universo financeiro",
        "CODACT 13 CAPILAR (Set/Out)",
        "CODEUT nulos",
        "FENTREGA nula",
    ],
    "valor": [
        len(df_raw),
        len(df_base),
        df_base["CODACT"].eq(13).sum(),
        df_base["CODEUT"].isna().sum(),
        df_base["FENTREGA"].isna().sum(),
    ],
})


,metrica,valor
0,Linhas raw,817764
1,Linhas universo financeiro,132215
2,CODACT 13 CAPILAR (Set/Out),6202
3,CODEUT nulos,0
4,FENTREGA nula,0


## 05. Danone — tarifa base do `011` pelo CSV; extra de 20 000 € apenas após associação ao `011`. CAPILAR `013` independente.


In [33]:
# ==========================
# 1. Preparar KILOSPO
# ==========================
df_danone = df_kilospo_raw.copy()

df_danone["referencia_chave"] = (
    normalizar_referencia(
        df_danone["PREFPE"]
    )
)

df_danone["PFEENT"] = normalizar_data(
    df_danone["PFEENT"]
)

df_danone["PCODCL"] = (
    df_danone["PCODCL"].astype("string").str.strip()
)
df_danone["CODACT"] = pd.to_numeric(
    df_danone["CODACT"], errors="coerce"
)
df_danone["Kg Neto Entregado"] = pd.to_numeric(
    df_danone["JDEKNS"].astype("string")
    .str.strip().str.replace(",", ".", regex=False),
    errors="coerce",
)

# ==========================
# 1.1. Filtrar actividade
# ==========================
df_danone = df_danone.loc[
    df_danone["PFEENT"].dt.year.eq(ANO)
    & df_danone["CODACT"].eq(11)
].copy()

if df_danone.empty:
    raise ValueError("INFKILP sem entregas válidas do CODACT 011 para 2026.")

# Referências livres ou códigos especiais ficam no diagnóstico,
# sem criar associações financeiras automáticas.

# ==========================
# 1.2. Classificar clientes por data
# ==========================
df_clientes = df_clientes_raw.copy()
df_clientes["pcodcl"] = df_clientes["pcodcl"].astype("string").str.strip()
df_clientes["tipo_local"] = df_clientes["tipo_local"].astype("string").str.strip()
df_clientes["ruta_wh"] = df_clientes["ruta_wh"].astype("string").str.strip()
df_clientes["data_inicio"] = pd.to_datetime(df_clientes["data_inicio"], errors="coerce")
df_clientes["data_fim"] = pd.to_datetime(df_clientes["data_fim"], errors="coerce")

if df_clientes["pcodcl"].isna().any() or df_clientes["data_inicio"].isna().any():
    raise ValueError("Tabela de clientes Danone com chave ou data de início inválida.")

if (df_clientes["data_fim"].notna() & df_clientes["data_fim"].lt(df_clientes["data_inicio"])).any():
    raise ValueError("Tabela de clientes Danone com datas invertidas.")

clientes_ord = df_clientes.sort_values(["pcodcl", "data_inicio"])
max_fim = clientes_ord["data_fim"].fillna(pd.Timestamp("2262-04-11"))
if (clientes_ord["data_inicio"] <= max_fim.groupby(clientes_ord["pcodcl"]).shift()).fillna(False).any():
    raise ValueError("Períodos de classificação Danone sobrepostos por PCODCL.")

df_danone["tipo_local"] = pd.Series(pd.NA, index=df_danone.index, dtype="string")
df_danone["ruta_wh"] = pd.Series(pd.NA, index=df_danone.index, dtype="string")
cliente_atribuido = pd.Series(False, index=df_danone.index)

for cliente in df_clientes.itertuples(index=False):
    mask = (
        df_danone["PCODCL"].eq(cliente.pcodcl)
        & df_danone["PFEENT"].ge(cliente.data_inicio)
    ).fillna(False)
    if pd.notna(cliente.data_fim):
        mask &= df_danone["PFEENT"].le(cliente.data_fim).fillna(False)
    if (cliente_atribuido & mask).any():
        raise ValueError("Várias classificações aplicáveis ao mesmo cliente Danone.")
    df_danone.loc[mask, "tipo_local"] = cliente.tipo_local
    df_danone.loc[mask, "ruta_wh"] = cliente.ruta_wh
    cliente_atribuido |= mask

# ==========================
# 2. Preparar tarifas
# ==========================
df_tarifas = df_tarifas_raw.copy()

df_tarifas["tipo_local"] = (
    df_tarifas["tipo_local"]
    .astype("string")
    .str.strip()
)

df_tarifas["modelo_ingresso"] = (
    df_tarifas["modelo_ingresso"]
    .astype("string")
    .str.strip()
)

df_tarifas["tarifa_base"] = (
    pd.to_numeric(
        df_tarifas["tarifa_base"].astype("string")
        .str.strip().str.replace(",", ".", regex=False),
        errors="coerce",
    )
)

df_tarifas["data_inicio"] = (
    pd.to_datetime(
        df_tarifas["data_inicio"],
        errors="coerce",
    )
)

df_tarifas["data_fim"] = (
    pd.to_datetime(
        df_tarifas["data_fim"],
        errors="coerce",
    )
)

if (
    df_tarifas["tipo_local"].isna().any()
    or df_tarifas["data_inicio"].isna().any()
):
    raise ValueError(
        "Existem tarifas sem tipo_local ou data_inicio."
    )

mask_intervalo_invalido = (
    df_tarifas["data_fim"].notna()
    & df_tarifas["data_fim"].lt(
        df_tarifas["data_inicio"]
    )
)

if mask_intervalo_invalido.any():
    raise ValueError(
        "Existem tarifas com data_fim anterior a data_inicio."
    )

# ==========================
# 3. Validar sobreposição
# ==========================
tarifas_ord = (
    df_tarifas
    .sort_values(
        ["tipo_local", "data_inicio"]
    )
    .copy()
)

tarifas_ord["fim_validacao"] = (
    tarifas_ord["data_fim"]
    .fillna(pd.Timestamp.max.normalize())
)

tarifas_ord["fim_anterior"] = (
    tarifas_ord
    .groupby("tipo_local")[
        "fim_validacao"
    ]
    .shift(1)
)

mask_sobreposicao = (
    tarifas_ord["fim_anterior"].notna()
    & tarifas_ord["data_inicio"].le(
        tarifas_ord["fim_anterior"]
    )
)

if mask_sobreposicao.any():
    tipos = (
        tarifas_ord.loc[
            mask_sobreposicao,
            "tipo_local",
        ]
        .dropna()
        .unique()
        .tolist()
    )
    raise ValueError(
        "Existem períodos tarifários sobrepostos: "
        f"{tipos}"
    )

# ==========================
# 4. Associar tarifa por data
# ==========================
df_danone["modelo_ingresso"] = pd.Series(
    pd.NA,
    index=df_danone.index,
    dtype="string",
)

df_danone["tarifa_base"] = pd.Series(
    pd.NA,
    index=df_danone.index,
    dtype="Float64",
)

df_danone["tarifa_data_inicio"] = pd.NaT
df_danone["tarifa_data_fim"] = pd.NaT

tarifa_atribuida = pd.Series(
    False,
    index=df_danone.index,
    dtype="bool",
)

for tarifa in df_tarifas.itertuples(index=False):
    mask_tarifa = (
        df_danone["tipo_local"].eq(
            tarifa.tipo_local
        )
        & df_danone["PFEENT"].ge(
            tarifa.data_inicio
        )
    ).fillna(False)

    if pd.notna(tarifa.data_fim):
        mask_tarifa &= (
            df_danone["PFEENT"].le(
                tarifa.data_fim
            )
            .fillna(False)
        )

    if (tarifa_atribuida & mask_tarifa).any():
        raise ValueError(
            "Mais de uma tarifa aplicável à mesma entrega Danone."
        )

    df_danone.loc[
        mask_tarifa,
        "modelo_ingresso",
    ] = tarifa.modelo_ingresso

    df_danone.loc[
        mask_tarifa,
        "tarifa_base",
    ] = tarifa.tarifa_base

    df_danone.loc[
        mask_tarifa,
        "tarifa_data_inicio",
    ] = tarifa.data_inicio

    df_danone.loc[
        mask_tarifa,
        "tarifa_data_fim",
    ] = tarifa.data_fim

    tarifa_atribuida |= mask_tarifa

# ==========================
# 5. Ingresso base
# ==========================
df_danone["ingresso_danone_base"] = (
    pd.Series(
        pd.NA,
        index=df_danone.index,
        dtype="Float64",
    )
)

mask_ton = (
    df_danone["modelo_ingresso"]
    .eq("POR_TONELADA")
    .fillna(False)
)

mask_entrega = (
    df_danone["modelo_ingresso"]
    .eq("POR_ENTREGA")
    .fillna(False)
)

mask_sem = (
    df_danone["modelo_ingresso"]
    .eq("SEM_INGRESSO")
    .fillna(False)
)

df_danone.loc[
    mask_ton,
    "ingresso_danone_base",
] = (
    df_danone.loc[
        mask_ton,
        "Kg Neto Entregado",
    ]
    / 1000
    * df_danone.loc[
        mask_ton,
        "tarifa_base",
    ]
)

df_danone.loc[
    mask_entrega,
    "ingresso_danone_base",
] = (
    df_danone.loc[
        mask_entrega,
        "tarifa_base",
    ]
)

df_danone.loc[
    mask_sem,
    "ingresso_danone_base",
] = 0.0

# ==========================
# 5.1. Tarifa fixa só uma vez por entrega
# ==========================
chave_entrega = ["PCODCL", "referencia_chave", "PFEENT"]
repetida_entrega = df_danone.duplicated(chave_entrega, keep="first")
df_danone.loc[
    mask_entrega & repetida_entrega,
    "ingresso_danone_base",
] = 0.0

if (mask_ton & df_danone["Kg Neto Entregado"].isna()).any():
    raise ValueError("Existem entregas por tonelada sem JDEKNS numérico.")

if ((mask_ton | mask_entrega) & df_danone["tarifa_base"].isna()).any():
    raise ValueError("Existem entregas com tarifa obrigatória em falta.")

# ==========================
# 5.2. Identificar CAPILAR sem tarifa unitária
# ==========================
mask_capilar = df_danone["modelo_ingresso"].eq("CAPILAR").fillna(False)
# O ingresso CAPILAR é atribuído depois da associação a CODEUT.

# ==========================
# 6. Preparar base sem extra
# ==========================
df_danone["mes_periodo"] = df_danone["PFEENT"].dt.to_period("M")
df_danone["extra_danone"] = 0.0
df_danone["ingresso_danone"] = df_danone["ingresso_danone_base"]

# ==========================
# 7. Estado do cálculo
# ==========================
condicoes = [
    (
        df_danone["modelo_ingresso"]
        .eq("POR_TONELADA")
        & df_danone["ingresso_danone"]
        .notna()
    ).fillna(False).to_numpy(dtype=bool),

    (
        df_danone["modelo_ingresso"]
        .eq("POR_ENTREGA")
        & df_danone["ingresso_danone"]
        .notna()
    ).fillna(False).to_numpy(dtype=bool),

    df_danone["modelo_ingresso"]
    .eq("SEM_INGRESSO")
    .fillna(False)
    .to_numpy(dtype=bool),

    df_danone["modelo_ingresso"]
    .eq("CAPILAR")
    .fillna(False)
    .to_numpy(dtype=bool),

    df_danone["modelo_ingresso"]
    .eq("POR_DEFINIR")
    .fillna(False)
    .to_numpy(dtype=bool),

    df_danone["modelo_ingresso"]
    .isna()
    .to_numpy(dtype=bool),
]

escolhas = [
    "CALCULADO_POR_TONELADA",
    "CALCULADO_POR_ENTREGA",
    "SEM_INGRESSO",
    "CAPILAR",
    "POR_DEFINIR",
    "SEM_TARIFA",
]

df_danone["estado_calculo"] = (
    np.select(
        condicoes,
        escolhas,
        default="REVER",
    )
)

# ==========================
# 8. Validar tarifa base do CSV
# ==========================
validacao_danone_base_mes = (
    df_danone.groupby("mes_periodo", as_index=False)
    .agg(
        linhas=("referencia_chave", "size"),
        ingresso_tarifa=("ingresso_danone_base", "sum"),
    )
)
validacao_danone_base_mes


,mes_periodo,linhas,ingresso_tarifa
0,2026-08,5159,149023.325441
1,2026-09,5398,146259.360449
2,2026-10,5503,146788.702701


In [34]:
df_danone

,ENTORNO,PREFPE,PRESPE,PFEENT,CODACT,PCODCL,PNOMCL,PRUTA,PCATCL,CODTLI,...,ruta_wh,modelo_ingresso,tarifa_base,tarifa_data_inicio,tarifa_data_fim,ingresso_danone_base,mes_periodo,extra_danone,ingresso_danone,estado_calculo
0,FGE50PTG,5023467410 415599852,4110478,2026-08-01,11.0,350148119,PINGO DOCE AZAMBUJA DIST. ALIM,2001,DI2,STD,...,Plataforma A,POR_TONELADA,14.434323,2026-01-01,NaT,164.861678,2026-08,0.0,164.861678,CALCULADO_POR_TONELADA
1,FGE50PTG,5023454997 415599852,4110478,2026-08-01,11.0,350148119,PINGO DOCE AZAMBUJA DIST. ALIM,2001,DI2,STD,...,Plataforma A,POR_TONELADA,14.434323,2026-01-01,NaT,0.0,2026-08,0.0,0.0,CALCULADO_POR_TONELADA
2,FGE50PTG,5023467410 415599852,4110536,2026-08-01,11.0,350148119,PINGO DOCE AZAMBUJA DIST. ALIM,2002,DI2,STD,...,Plataforma A,POR_TONELADA,14.434323,2026-01-01,NaT,137.536436,2026-08,0.0,137.536436,CALCULADO_POR_TONELADA
3,FGE50PTG,5023459924 415605419,4110496,2026-08-01,11.0,350152530,MODELO CONT.HIP.SA-MAIA,2600,DI2,STD,...,Plataforma A,POR_TONELADA,31.493069,2026-01-01,NaT,0.0,2026-08,0.0,0.0,CALCULADO_POR_TONELADA
4,FGE50PTG,5023478721 415605418,4110496,2026-08-01,11.0,350152530,MODELO CONT.HIP.SA-MAIA,2600,DI2,STD,...,Plataforma A,POR_TONELADA,31.493069,2026-01-01,NaT,0.0,2026-08,0.0,0.0,CALCULADO_POR_TONELADA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16055,FGE50PTG,5024205291 416285332,4126938,2026-10-31,11.0,350476819,AUCHAN - AZAMBUJA,6800,DI2,STD,...,Plataforma A,SEM_INGRESSO,0.0,2026-01-01,NaT,0.0,2026-10,0.0,0.0,SEM_INGRESSO
16056,FGE50PTG,5024201615 416297843,4126938,2026-10-31,11.0,350476819,AUCHAN - AZAMBUJA,6800,DI2,STD,...,Plataforma A,SEM_INGRESSO,0.0,2026-01-01,NaT,0.0,2026-10,0.0,0.0,SEM_INGRESSO
16057,FGE50PTG,5024210978 416309368,4127080,2026-10-29,11.0,350141432,CATERING POR.,1805,DDA,STD,...,Directos PV,POR_TONELADA,91.85262,2026-01-01,NaT,1.025075,2026-10,0.0,1.025075,CALCULADO_POR_TONELADA
16058,FGE50PTG,5024191237 416317662,4127327,2026-10-26,11.0,350150470,MAKRO - PLATAFORMA,2100,DI2,STD,...,Plataforma B,POR_TONELADA,52.488448,2026-01-01,NaT,0.13437,2026-10,0.0,0.13437,CALCULADO_POR_TONELADA


## 06. Associação Danone e reconciliação

In [35]:
# ==========================
# 1. Calcular ingresso base por entrega
# ==========================
# Tarifas CAPILAR no CSV não integram CODACT 011.
danone_chave = (
    df_danone.loc[
        df_danone["referencia_chave"].notna()
        & df_danone["PFEENT"].notna()
    ]
    .groupby(["referencia_chave", "PFEENT"], as_index=False)
    .agg(
        ingresso_danone_base_total=(
            "ingresso_danone_base",
            lambda x: x.sum(min_count=1),
        ),
        e_capilar=("modelo_ingresso", lambda x: x.eq("CAPILAR").any()),
    )
)

# ==========================
# 2. Associar ao CODACT 011
# ==========================
df = df_base.copy()
df = df.merge(
    danone_chave,
    left_on=["referencia_chave", "FENTREGA"],
    right_on=["referencia_chave", "PFEENT"],
    how="left",
    validate="many_to_one",
)

# ==========================
# 3. Seleccionar uma linha por entrega
# ==========================
# A chave da entrega é referência + data. A tarifa só é atribuída
# a uma linha do CODACT 011 para evitar repetições.
tem_match = (
    df["CODACT"].eq(11)
    & df["ingresso_danone_base_total"].notna()
)
destino_preenchido = (
    df["CODACT"].eq(11)
    & df["DESTINO"].astype("string").fillna("").str.strip().ne("")
)
n_destinos = destino_preenchido.groupby(
    [df["referencia_chave"], df["FENTREGA"]],
    dropna=False,
).transform("sum")
linha_destino_unico = tem_match & destino_preenchido & n_destinos.eq(1)
linha_fallback = (
    tem_match & ~n_destinos.eq(1)
    & ~df.duplicated(["CODACT", "referencia_chave", "FENTREGA"], keep="last")
)
linha_ingresso_danone = linha_destino_unico | linha_fallback

# ==========================
# 4. Atribuir tarifa base ao CODACT 011
# ==========================
df["ingresso_danone_base"] = 0.0
df.loc[linha_ingresso_danone, "ingresso_danone_base"] = (
    df.loc[linha_ingresso_danone, "ingresso_danone_base_total"].astype(float)
)

# ==========================
# 5. Distribuir extra após associação
# ==========================
df["extra_danone"] = 0.0
mask_extra = (
    df["CODACT"].eq(11)
    & linha_ingresso_danone
    & df["ingresso_danone_base"].gt(0)
    & df["ano"].eq(ANO)
)
base_elegivel_mes = (
    df["ingresso_danone_base"].where(mask_extra)
    .groupby([df["ano"], df["mes"]])
    .transform("sum")
)
if EXTRA_DANONE_MENSAL:
    df.loc[mask_extra, "extra_danone"] = (
        df.loc[mask_extra, "ingresso_danone_base"]
        / base_elegivel_mes.loc[mask_extra]
        * EXTRA_DANONE_MENSAL
    )

df["ingresso_danone"] = (
    df["ingresso_danone_base"] + df["extra_danone"]
)

# ==========================
# 6. Reconciliar ingressos calculados
# ==========================
# A base sem correspondência não é somada a outra viagem.
# O relatório evidencia a diferença por mês.
chaves_associadas = (
    df.loc[linha_ingresso_danone, ["referencia_chave", "FENTREGA"]]
    .drop_duplicates()
    .rename(columns={"FENTREGA": "PFEENT"})
    .assign(ASSOCIADO=True)
)
danone_chave = danone_chave.merge(
    chaves_associadas,
    on=["referencia_chave", "PFEENT"],
    how="left",
    validate="one_to_one",
)
danone_chave["ASSOCIADO"] = danone_chave["ASSOCIADO"].fillna(False).astype(bool)

# ==========================
# 6.2. Rateio por entregas e rotas CAPILAR
# ==========================
df["ingresso_capilar"] = 0.0
mask_capilar_financeiro = (
    df["CODACT"].eq(13)
    & df["FENTREGA"].dt.year.eq(ANO)
    & df["FENTREGA"].dt.month.isin(INGRESSO_CAPILAR_MENSAL)
)

# Um movimento com mesma referência, data e CODEUT conta uma só entrega.
# Se a referência for nula, a transacção não pode ser rateada em segurança.
df_capilar_rotas = df.loc[
    mask_capilar_financeiro,
    ["_linha_id", "FENTREGA", "CODEUT", "referencia_chave"],
].copy()
df_capilar_rotas["MES"] = df_capilar_rotas["FENTREGA"].dt.month

if df_capilar_rotas[["FENTREGA", "CODEUT", "referencia_chave"]].isna().any().any():
    raise ValueError("CODACT 013: existem entregas sem data, CODEUT ou referência.")

for mes, valor_mensal in INGRESSO_CAPILAR_MENSAL.items():
    entregas_mes = df_capilar_rotas.loc[df_capilar_rotas["MES"].eq(mes)].copy()
    if entregas_mes.empty:
        raise ValueError(f"CAPILAR {mes:02d}: não existem entregas CODACT 013.")

    entregas_unicas = entregas_mes.drop_duplicates(
        ["FENTREGA", "CODEUT", "referencia_chave"],
        keep="first",
    )
    valor_por_entrega = valor_mensal / len(entregas_unicas)
    mask_destino = df["_linha_id"].isin(entregas_unicas["_linha_id"])
    df.loc[mask_destino, "ingresso_capilar"] = valor_por_entrega

    total_mes = df.loc[
        df["CODACT"].eq(13) & df["FENTREGA"].dt.month.eq(mes),
        "ingresso_capilar",
    ].sum()
    if abs(total_mes - valor_mensal) > TOLERANCIA_FINANCEIRA:
        raise ValueError(
            f"CAPILAR {mes:02d}: obtidos {total_mes:.2f} €, "
            f"esperados {valor_mensal:.2f} €."
        )

# O ingresso Danone é exclusivo do CODACT 011.
df.loc[df["CODACT"].eq(13), "ingresso_danone"] = 0.0

validacao_capilar_mensal = (
    df.loc[mask_capilar_financeiro]
    .assign(MES=lambda x: x["FENTREGA"].dt.month)
    .groupby("MES", as_index=False)
    .agg(
        ingresso_capilar=("ingresso_capilar", "sum"),
        rotas=("CODEUT", "nunique"),
        linhas=("_linha_id", "size"),
    )
)

# ==========================
# 6.3. Validar separação por actividade
# ==========================
if df.loc[df["CODACT"].eq(13), "ingresso_danone"].fillna(0).ne(0).any():
    raise ValueError("CODACT 013 recebeu indevidamente ingresso Danone 011.")
if df.loc[df["CODACT"].eq(11), "ingresso_capilar"].ne(0).any():
    raise ValueError("CODACT 011 recebeu indevidamente ingresso CAPILAR.")

# ==========================
# 7. Limpar colunas auxiliares
# ==========================

df = df.drop(
    columns=[
        "PFEENT",
        "ingresso_danone_base_total",
        "e_capilar",
    ],
    errors="ignore",
)


# ==========================
# 8. Ingresso total
# ==========================

df["total_ingresso"] = (
    df["INGRESODT"].fillna(0.0)
    + df["ingresso_danone"].fillna(0.0)
    + df["ingresso_capilar"].fillna(0.0)
)

# ==========================
# 9. Validar extra após associação
# ==========================
validacao_danone_mes = (
    df.loc[df["CODACT"].eq(11)]
    .groupby(["ano", "mes"], as_index=False)
    .agg(
        ingresso_tarifa=("ingresso_danone_base", "sum"),
        extra=("extra_danone", "sum"),
        ingresso_final=("ingresso_danone", "sum"),
    )
)
validacao_danone_mes


/var/folders/8l/z5bjd49s42b04x3cy5cc16mh0000gn/T/ipykernel_62831/3110374985.py:107: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  danone_chave["ASSOCIADO"] = danone_chave["ASSOCIADO"].fillna(False).astype(bool)


,ano,mes,ingresso_tarifa,extra,ingresso_final
0,2026,1,0.000000,0.0,0.000000
1,2026,2,0.000000,0.0,0.000000
2,2026,3,0.000000,0.0,0.000000
3,2026,4,0.000000,0.0,0.000000
4,2026,5,0.000000,0.0,0.000000
5,2026,6,0.000000,0.0,0.000000
6,2026,7,0.000000,0.0,0.000000
7,2026,8,141721.479707,20000.0,161721.479707
8,2026,9,135703.986869,20000.0,155703.986869
9,2026,10,139779.428534,20000.0,159779.428534


In [36]:
# ==========================
# 1. Âmbito de validação
# ==========================
danone_validacao = (
    danone_chave.loc[
        danone_chave["PFEENT"]
        .dt.month
        .le(MES_VALIDACAO_DANONE_MAX)
    ]
    .copy()
)

base_validacao = (
    df_base.loc[
        df_base["CODACT"].eq(11)
        & df_base["FENTREGA"]
        .dt.month
        .le(MES_VALIDACAO_DANONE_MAX)
    ]
    .copy()
)

raw_validacao = (
    df_raw_check.loc[
        df_raw_check["FENTREGA"]
        .dt.month
        .le(MES_VALIDACAO_DANONE_MAX)
        | df_raw_check["FENTREGA"].isna()
    ]
    .copy()
)

# ==========================
# 2. Chaves disponíveis
# ==========================
chaves_base_exactas = set(
    zip(
        base_validacao["referencia_chave"],
        base_validacao["FENTREGA"],
    )
)

refs_base = set(
    base_validacao[
        "referencia_chave"
    ].dropna()
)

refs_raw = set(
    raw_validacao[
        "referencia_chave"
    ].dropna()
)

refs_codact13 = set(
    raw_validacao.loc[
        raw_validacao["CODACT"].eq(13),
        "referencia_chave",
    ].dropna()
)

# ==========================
# 3. Classificar diferença
# ==========================
def classificar_danone(row):
    chave = (
        row["referencia_chave"],
        row["PFEENT"],
    )

    ref = row["referencia_chave"]

    if chave in chaves_base_exactas:
        return "ASSOCIADO_EXACTO"

    if ref in refs_codact13:
        return "EXCLUIDO_CODACT_13"

    if ref in refs_base:
        return "REFERENCIA_EXISTE_DATA_DIFERENTE"

    if ref in refs_raw:
        return "EXCLUIDO_OUTRO_FILTRO"

    return "FORA_INFORM27"

danone_validacao[
    "estado_reconciliacao"
] = danone_validacao.apply(
    classificar_danone,
    axis=1,
)

# ==========================
# 4. Resumo
# ==========================
validacao_danone = (
    danone_validacao
    .groupby(
        "estado_reconciliacao",
        as_index=False,
    )
    .agg(
        casos=(
            "referencia_chave",
            "size",
        ),
        referencias=(
            "referencia_chave",
            "nunique",
        ),
        ingresso_base=(
            "ingresso_danone_base_total",
            "sum",
        ),
        extra=(
            "ingresso_danone_base_total",
            lambda x: 0.0,
        ),
        ingresso=(
            "ingresso_danone_base_total",
            "sum",
        ),
    )
    .sort_values(
        "ingresso",
        ascending=False,
    )
    .reset_index(drop=True)
)

# ==========================
# 5. Reconciliação financeira real
# ==========================
df_validacao_danone = (
    df.loc[
        df["CODACT"].eq(11)
        & df["FENTREGA"]
        .dt.month
        .le(MES_VALIDACAO_DANONE_MAX)
    ]
    .copy()
)

total_danone_df = (
    df_validacao_danone[
        "ingresso_danone_base"
    ]
    .fillna(0.0)
    .sum()
)

total_danone_associado_exacto = (
    danone_validacao.loc[
        danone_validacao[
            "estado_reconciliacao"
        ].eq("ASSOCIADO_EXACTO"),
        "ingresso_danone_base_total",
    ]
    .fillna(0.0)
    .sum()
)

erro_reconciliacao_danone = (
    total_danone_df
    - total_danone_associado_exacto
)

# ==========================
# 6. Validar troço final
# ==========================
validacao_troco_danone = (
    df.loc[
        df["CODACT"].eq(11)
        & df["referencia_chave"].notna()
        & df["FENTREGA"].notna()
    ]
    .copy()
)

validacao_troco_danone[
    "destino_preenchido"
] = (
    validacao_troco_danone["DESTINO"]
    .astype("string")
    .fillna("")
    .str.strip()
    .ne("")
)

validacao_troco_danone[
    "n_destinos_chave"
] = (
    validacao_troco_danone
    .groupby(
        ["referencia_chave", "FENTREGA"]
    )["destino_preenchido"]
    .transform("sum")
)

casos_troco_danone_errados = int(
    (
        validacao_troco_danone[
            "ingresso_danone"
        ].notna()
        & validacao_troco_danone[
            "n_destinos_chave"
        ].eq(1)
        & ~validacao_troco_danone[
            "destino_preenchido"
        ]
    ).sum()
)

validacao_danone


,estado_reconciliacao,casos,referencias,ingresso_base,extra,ingresso
0,ASSOCIADO_EXACTO,4992,4992,417204.89511,0.0,417204.89511
1,EXCLUIDO_CODACT_13,9093,9093,17290.478668,0.0,17290.478668
2,FORA_INFORM27,947,947,6379.843879,0.0,6379.843879
3,REFERENCIA_EXISTE_DATA_DIFERENTE,38,38,1196.170934,0.0,1196.170934


## 07. Combustível

In [37]:
# ==========================
# 1. Mês da taxa
# ==========================
df["mes_taxa_combustivel"] = (
    df["FENTREGA"].dt.month
    + DESFASAMENTO_MES_COMBUSTIVEL
)

# ==========================
# 2. Taxa
# ==========================
def obter_taxa(codact, mes):
    if pd.isna(codact) or pd.isna(mes):
        return np.nan

    mes = int(mes)

    if mes < 1 or mes > 12:
        return np.nan

    return taxas_ingresso.get(
        int(codact),
        {},
    ).get(
        mes,
        np.nan,
    )


def taxa_configurada_no_periodo(codact, mes):
    if pd.isna(codact) or pd.isna(mes):
        return False

    taxas_codact = taxas_ingresso.get(
        int(codact),
        {},
    )

    if not taxas_codact:
        return False

    mes = int(mes)
    meses = sorted(taxas_codact)

    return (
        min(meses)
        <= mes
        <= max(meses)
    )


df["taxa_combustivel"] = [
    obter_taxa(codact, mes)
    for codact, mes in zip(
        df["CODACT"],
        df["mes_taxa_combustivel"],
    )
]

df["taxa_combustivel"] = (
    pd.to_numeric(
        df["taxa_combustivel"],
        errors="coerce",
    )
)

# ==========================
# 3. Base combustível
# ==========================
df["base_combustivel"] = (
    pd.to_numeric(
        df["INGRESODT"],
        errors="coerce",
    )
)

mask_codact11 = (
    df["CODACT"].eq(11)
)

df.loc[
    mask_codact11,
    "base_combustivel",
] = (
    pd.to_numeric(
        df.loc[
            mask_codact11,
            "ingresso_danone",
        ],
        errors="coerce",
    )
)

# ==========================
# 4. Validar taxa em falta
# ==========================
df["taxa_combustivel_esperada"] = [
    taxa_configurada_no_periodo(
        codact,
        mes,
    )
    for codact, mes in zip(
        df["CODACT"],
        df["mes_taxa_combustivel"],
    )
]

df["taxa_combustivel_em_falta"] = (
    df["taxa_combustivel_esperada"]
    & df["taxa_combustivel"].isna()
    & df["base_combustivel"]
    .fillna(0.0)
    .ne(0.0)
)

# ==========================
# 5. Acerto combustível
# ==========================
df["acerto_combustivel"] = 0.0

mask_taxa = (
    df["taxa_combustivel"].notna()
    & df["base_combustivel"].notna()
)

df.loc[
    mask_taxa,
    "acerto_combustivel",
] = (
    df.loc[
        mask_taxa,
        "base_combustivel",
    ]
    * df.loc[
        mask_taxa,
        "taxa_combustivel",
    ]
)

df["total_ingresso_c"] = (
    df["total_ingresso"]
    + df["acerto_combustivel"]
)


## 08. Capacidade e coordenadas

In [38]:
# ==========================
# 1. Capacidade
# ==========================

df["CAMION_CAPACIDAD_NUM"] = (
    pd.to_numeric(
        df["CAMION_CAPACIDAD"],
        errors="coerce",
    )
)

df["capacidade_norm"] = (
    df["CAMION_CAPACIDAD_NUM"]
    .map(MAPA_CAPACIDADE)
)

capacidade_codeut = (
    df.loc[
        df["CODEUT"].notna()
        & df["capacidade_norm"].notna()
    ]
    .groupby("CODEUT")[
        "capacidade_norm"
    ]
    .agg(moda_segura)
)

mask_sem_capacidade = (
    df["CODEUT"].notna()
    & df["capacidade_norm"].isna()
)

df.loc[
    mask_sem_capacidade,
    "capacidade_norm",
] = (
    df.loc[
        mask_sem_capacidade,
        "CODEUT",
    ]
    .map(capacidade_codeut)
)


# ==========================
# 2. Paletes por CODEUT
# ==========================

paletes_codeut = (
    df.loc[
        df["CODEUT"].notna()
    ]
    .groupby("CODEUT")[
        "PALETS"
    ]
    .sum(min_count=1)
)


# ==========================
# 3. Corrigir capacidade 33
# ==========================

df["paletes_codeut"] = (
    df["CODEUT"]
    .map(paletes_codeut)
)

mask_capacidade_33 = (
    df["capacidade_norm"].eq(33)
)

df.loc[
    mask_capacidade_33
    & df["paletes_codeut"].gt(40)
    & df["paletes_codeut"].le(55),
    "capacidade_norm",
] = 55

df.loc[
    mask_capacidade_33
    & df["paletes_codeut"].gt(55),
    "capacidade_norm",
] = 66


# ==========================
# 4. Paletes para ocupação
# ==========================

df["paletes_ocupacao_codeut"] = (
    df["paletes_codeut"]
    .clip(upper=66)
)


# ==========================
# 5. Dados válidos
# ==========================

df["dados_validos"] = (
    df["CODEUT"].notna()
    & df["FENTREGA"].notna()
    & df["capacidade_norm"].notna()
    & df["paletes_ocupacao_codeut"].notna()
)


In [39]:
# ==========================
# 1. Preparar coordenadas
# ==========================
coords = coords_raw.copy()

coords["POINT_X"] = pd.to_numeric(
    coords["POINT_X"]
    .astype("string")
    .str.strip()
    .str.replace(",", ".", regex=False),
    errors="coerce",
)

coords["POINT_Y"] = pd.to_numeric(
    coords["POINT_Y"]
    .astype("string")
    .str.strip()
    .str.replace(",", ".", regex=False),
    errors="coerce",
)

coords["CP_chave"] = normalizar_cp(
    coords["CP"]
)

coords["CP_parte"] = (
    coords["CP_chave"]
    .str[:4]
)

df["CPOSTAL_chave"] = normalizar_cp(
    df["CPOSTAL"]
)

df["CPOSTAD_chave"] = normalizar_cp(
    df["CPOSTAD"]
)

df["CPOSTAL_parte"] = (
    df["CPOSTAL_chave"]
    .str[:4]
)

df["CPOSTAD_parte"] = (
    df["CPOSTAD_chave"]
    .str[:4]
)

# ==========================
# 2. CP exacto
# ==========================
coords_cp = (
    coords.dropna(
        subset=[
            "CP_chave",
            "POINT_X",
            "POINT_Y",
        ]
    )
    .groupby(
        "CP_chave",
        as_index=False,
    )[
        ["POINT_X", "POINT_Y"]
    ]
    .mean()
)

origem = coords_cp.rename(
    columns={
        "CP_chave": "CPOSTAL_chave",
        "POINT_X": "longitude_origem",
        "POINT_Y": "latitude_origem",
    }
)

destino = coords_cp.rename(
    columns={
        "CP_chave": "CPOSTAD_chave",
        "POINT_X": "longitude_destino",
        "POINT_Y": "latitude_destino",
    }
)

df = df.merge(
    origem,
    on="CPOSTAL_chave",
    how="left",
    validate="many_to_one",
)

df = df.merge(
    destino,
    on="CPOSTAD_chave",
    how="left",
    validate="many_to_one",
)

# ==========================
# 3. Fallback 4 dígitos
# ==========================
centroid = (
    coords.dropna(
        subset=[
            "CP_parte",
            "POINT_X",
            "POINT_Y",
        ]
    )
    .groupby(
        "CP_parte",
        as_index=False,
    )[
        ["POINT_X", "POINT_Y"]
    ]
    .mean()
)

centroid_origem = centroid.rename(
    columns={
        "CP_parte": "CPOSTAL_parte",
        "POINT_X": "longitude_origem_fb",
        "POINT_Y": "latitude_origem_fb",
    }
)

centroid_destino = centroid.rename(
    columns={
        "CP_parte": "CPOSTAD_parte",
        "POINT_X": "longitude_destino_fb",
        "POINT_Y": "latitude_destino_fb",
    }
)

df = df.merge(
    centroid_origem,
    on="CPOSTAL_parte",
    how="left",
    validate="many_to_one",
)

df = df.merge(
    centroid_destino,
    on="CPOSTAD_parte",
    how="left",
    validate="many_to_one",
)

df["longitude_origem"] = (
    df["longitude_origem"]
    .fillna(
        df["longitude_origem_fb"]
    )
)

df["latitude_origem"] = (
    df["latitude_origem"]
    .fillna(
        df["latitude_origem_fb"]
    )
)

df["longitude_destino"] = (
    df["longitude_destino"]
    .fillna(
        df["longitude_destino_fb"]
    )
)

df["latitude_destino"] = (
    df["latitude_destino"]
    .fillna(
        df["latitude_destino_fb"]
    )
)

df = df.drop(
    columns=[
        "longitude_origem_fb",
        "latitude_origem_fb",
        "longitude_destino_fb",
        "latitude_destino_fb",
    ],
    errors="ignore",
)

## 09. Custos, viagens e rentabilidade

In [40]:
# ==========================
# 1. Preparar valores
# ==========================
df["PALETS"] = pd.to_numeric(
    df["PALETS"],
    errors="coerce",
).fillna(0.0)

df["COSTEDT"] = pd.to_numeric(
    df["COSTEDT"],
    errors="coerce",
).fillna(0.0)

# ==========================
# 2. Paletes viagem/mês
# ==========================
paletes_codeut_mes = (
    df.groupby(
        [
            "mes_estrutura",
            "CODEUT",
        ],
        dropna=False,
    )["PALETS"]
    .transform("sum")
)

mask_estrutura = (
    df["mes_estrutura"].notna()
    & df["CODEUT"].notna()
    & paletes_codeut_mes.gt(0)
)

# ==========================
# 3. Viagens mensais
# ==========================
n_codeut_mes = (
    df.loc[
        mask_estrutura
    ]
    .groupby(
        "mes_estrutura"
    )["CODEUT"]
    .nunique()
)

df["n_viagens_estrutura_mes"] = (
    df["mes_estrutura"]
    .map(n_codeut_mes)
)

df["custo_estrutura_viagem"] = (
    CUSTO_ESTRUTURA_MENSAL
    / df["n_viagens_estrutura_mes"]
)

# ==========================
# 4. Rateio estrutura
# ==========================
df["custo_estrutura"] = 0.0

df.loc[
    mask_estrutura,
    "custo_estrutura",
] = (
    df.loc[
        mask_estrutura,
        "custo_estrutura_viagem",
    ]
    * df.loc[
        mask_estrutura,
        "PALETS",
    ]
    / paletes_codeut_mes.loc[
        mask_estrutura
    ]
)

# ==========================
# 5. Custo e margem
# ==========================
df["custo_total_c"] = (
    df["COSTEDT"]
    + df["custo_estrutura"]
)

df["margem"] = (
    df["total_ingresso_c"]
    - df["custo_total_c"]
)

df["custo_por_palete"] = (
    dividir_seguro(
        df["custo_total_c"],
        df["PALETS"],
    )
)

df["ingresso_por_palete"] = (
    dividir_seguro(
        df["total_ingresso_c"],
        df["PALETS"],
    )
)

df["margem_por_palete"] = (
    dividir_seguro(
        df["margem"],
        df["PALETS"],
    )
)

df["margem_pct"] = (
    dividir_seguro(
        df["margem"],
        df["total_ingresso_c"],
    )
    * 100
)

In [41]:
# ==========================
# 1. Tabela oficial de viagens
# ==========================
df_viagens = (
    df.loc[
        df["CODEUT"].notna()
    ]
    .groupby(
        "CODEUT",
        as_index=False,
    )
    .agg(
        data=("FENTREGA", "min"),
        mes_estrutura=(
            "mes_estrutura",
            "min",
        ),
        TRANSPORTISTA=(
            "TRANSPORTISTA",
            "first",
        ),
        LOCORIGEN=(
            "LOCORIGEN",
            "first",
        ),
        LOCDESTINO=(
            "LOCDESTINO",
            "first",
        ),
        paletes=("PALETS", "sum"),
        paletes_ocupacao=(
            "paletes_ocupacao_codeut",
            "first",
        ),
        peso_kg=("PESO_BRUTO", "sum"),
        custo_transporte=(
            "COSTEDT",
            "sum",
        ),
        custo_estrutura=(
            "custo_estrutura",
            "sum",
        ),
        custo_total=(
            "custo_total_c",
            "sum",
        ),
        ingresso_total=(
            "total_ingresso_c",
            "sum",
        ),
        capacidade_original=(
            "CAMION_CAPACIDAD_NUM",
            moda_segura,
        ),
        capacidade_real=(
            "capacidade_norm",
            moda_segura,
        ),
    )
)

# ==========================
# 2. Estado ocupação
# ==========================
condicoes_ocupacao = [
    df_viagens["capacidade_real"]
    .isna()
    .to_numpy(dtype=bool),

    df_viagens["capacidade_real"]
    .le(0)
    .fillna(False)
    .to_numpy(dtype=bool),

    df_viagens["paletes_ocupacao"]
    .isna()
    .to_numpy(dtype=bool),
]

df_viagens["ocupacao_estado"] = (
    np.select(
        condicoes_ocupacao,
        [
            "SEM_CAPACIDADE",
            "CAPACIDADE_INVALIDA",
            "SEM_PALETES",
        ],
        default="OK",
    )
)

# ==========================
# 3. Ocupação
# ==========================
df_viagens["ocupacao_pct"] = (
    dividir_seguro(
        df_viagens["paletes_ocupacao"],
        df_viagens["capacidade_real"],
    )
    * 100
)

df_viagens.loc[
    ~df_viagens[
        "ocupacao_estado"
    ].eq("OK"),
    "ocupacao_pct",
] = np.nan

# ==========================
# 4. KPIs viagem
# ==========================
df_viagens["peso_ton"] = (
    df_viagens["peso_kg"]
    / 1000
)

df_viagens["custo_por_palete"] = (
    dividir_seguro(
        df_viagens["custo_total"],
        df_viagens["paletes"],
    )
)

df_viagens[
    "ingresso_por_palete"
] = dividir_seguro(
    df_viagens["ingresso_total"],
    df_viagens["paletes"],
)

df_viagens["custo_por_ton"] = (
    dividir_seguro(
        df_viagens["custo_total"],
        df_viagens["peso_ton"],
    )
)

df_viagens[
    "ingresso_por_ton"
] = dividir_seguro(
    df_viagens["ingresso_total"],
    df_viagens["peso_ton"],
)

df_viagens["margem"] = (
    df_viagens["ingresso_total"]
    - df_viagens["custo_total"]
)

df_viagens["margem_pct"] = (
    dividir_seguro(
        df_viagens["margem"],
        df_viagens["ingresso_total"],
    )
    * 100
)

# ==========================
# 5. Associar ao detalhe
# ==========================
df = df.merge(
    df_viagens[
        [
            "CODEUT",
            "capacidade_real",
            "ocupacao_pct",
            "ocupacao_estado",
        ]
    ],
    on="CODEUT",
    how="left",
    validate="many_to_one",
)


## 10. Validações finais

In [42]:
# ==========================
# 1. Estrutura mensal
# ==========================
validacao_estrutura = (
    df.loc[
        df["mes_estrutura"].isin(
            n_codeut_mes.index
        )
    ]
    .groupby(
        "mes_estrutura",
        as_index=False,
    )
    .agg(
        viagens=(
            "CODEUT",
            "nunique",
        ),
        estrutura=(
            "custo_estrutura",
            "sum",
        ),
    )
)

validacao_estrutura["esperado"] = (
    CUSTO_ESTRUTURA_MENSAL
)

validacao_estrutura["erro"] = (
    validacao_estrutura["estrutura"]
    - validacao_estrutura["esperado"]
)

erro_estrutura = (
    validacao_estrutura["erro"]
    .abs()
    .max()
)

if pd.isna(erro_estrutura):
    erro_estrutura = 0.0

# ==========================
# 2. Consistência detalhe/viagens
# ==========================
erro_custo = abs(
    df["custo_total_c"].sum()
    - df_viagens["custo_total"].sum()
)

erro_ingresso = abs(
    df["total_ingresso_c"].sum()
    - df_viagens["ingresso_total"].sum()
)

erro_margem = (
    (
        df["margem"]
        - (
            df["total_ingresso_c"]
            - df["custo_total_c"]
        )
    )
    .abs()
    .max()
)

if pd.isna(erro_margem):
    erro_margem = 0.0

# ==========================
# 3. Validações adicionais
# ==========================
taxas_combustivel_em_falta = int(
    df["taxa_combustivel_em_falta"]
    .sum()
)

linhas_fora_ano = int(
    (
        df["FENTREGA"].notna()
        & ~df["FENTREGA"]
        .dt.year
        .eq(ANO)
    ).sum()
)

# ==========================
# 4. Controlos finais
# ==========================
validacao_financeira = pd.DataFrame({
    "controlo": [
        "Danone df = ASSOCIADO_EXACTO",
        "Danone no troço final",
        "Taxa combustível em falta",
        "FENTREGA fora do ano",
        "Estrutura mensal",
        "Custo detalhe = viagens",
        "Ingresso detalhe = viagens",
        "CODEUT texto nan",
        "CODEUT único em viagens",
        "Ocupação OK > 100%",
        "Margem consistente",
        "CAPILAR Setembro = 93k",
        "CAPILAR Outubro = 85k",
    ],
    "valor": [
        abs(float(erro_reconciliacao_danone)),
        int(casos_troco_danone_errados),
        int(taxas_combustivel_em_falta),
        int(linhas_fora_ano),
        float(erro_estrutura),
        float(erro_custo),
        float(erro_ingresso),
        int(df["CODEUT"].eq("nan").sum()),
        int(
            df_viagens["CODEUT"]
            .duplicated()
            .sum()
        ),
        int(
            df_viagens.loc[
                df_viagens[
                    "ocupacao_estado"
                ].eq("OK"),
                "ocupacao_pct",
            ]
            .gt(100 + 1e-9)
            .sum()
        ),
        float(erro_margem),
        abs(df.loc[df["mes"].eq(9), "ingresso_capilar"].sum() - 93_000.0),
        abs(df.loc[df["mes"].eq(10), "ingresso_capilar"].sum() - 85_000.0),
    ],
    "tolerancia": [
        TOLERANCIA_FINANCEIRA,
        0,
        0,
        0,
        TOLERANCIA_FINANCEIRA,
        TOLERANCIA_FINANCEIRA,
        TOLERANCIA_FINANCEIRA,
        0,
        0,
        0,
        TOLERANCIA_FINANCEIRA,
        TOLERANCIA_FINANCEIRA,
        TOLERANCIA_FINANCEIRA,
    ],
})

validacao_financeira["estado"] = (
    np.where(
        validacao_financeira["valor"]
        <= validacao_financeira[
            "tolerancia"
        ],
        "OK",
        "ERRO",
    )
)

validacao_financeira


,controlo,valor,tolerancia,estado
0,Danone df = ASSOCIADO_EXACTO,0.000000e+00,0.01,OK
1,Danone no troço final,2.730000e+03,0.00,ERRO
2,Taxa combustível em falta,0.000000e+00,0.00,OK
3,FENTREGA fora do ano,0.000000e+00,0.00,OK
4,Estrutura mensal,1.455192e-11,0.01,OK
5,Custo detalhe = viagens,0.000000e+00,0.01,OK
6,Ingresso detalhe = viagens,9.313226e-10,0.01,OK
7,CODEUT texto nan,0.000000e+00,0.00,OK
8,CODEUT único em viagens,0.000000e+00,0.00,OK
9,Ocupação OK > 100%,2.661000e+03,0.00,ERRO


## 11. Dataset final e exportação

In [43]:
# ==========================
# 1. Colunas finais
# ==========================
colunas_finais = [
    "REFERENCIA",
    "referencia_chave",
    "FENTREGA",
    "FCARGA",
    "CODEUT",
    "capacidade_norm",
    "paletes_codeut",
    "paletes_ocupacao_codeut",
    "capacidade_real",
    "ocupacao_pct",
    "ocupacao_estado",
    "PALETS",
    "PESO_BRUTO",
    "INGRESODT",
    "ingresso_danone_base",
    "extra_danone",
    "ingresso_danone",
    "ingresso_capilar",
    "total_ingresso",
    "taxa_combustivel",
    "taxa_combustivel_esperada",
    "taxa_combustivel_em_falta",
    "base_combustivel",
    "acerto_combustivel",
    "total_ingresso_c",
    "COSTEDT",
    "custo_estrutura",
    "custo_total_c",
    "margem",
    "margem_pct",
    "custo_por_palete",
    "ingresso_por_palete",
    "margem_por_palete",
    "PROV_ORIGEN",
    "LOCORIGEN",
    "PROV_DESTINO",
    "LOCDESTINO",
    "CPOSTAL",
    "CPOSTAD",
    "longitude_origem",
    "latitude_origem",
    "longitude_destino",
    "latitude_destino",
    "TRANSPORTISTA",
    "week_day",
    "week_number",
    "ano",
    "mes",
    "mes_nome",
    "data",
    "mes_estrutura",
    "dados_validos",
    "LOCCAR",
    "LUGARCARGA",
    "LOCDES",
    "LUGARDESCARGA",
    "TRACTORA",
    "CODEDT",
    "CODACT",
    "TIPOCLIENTE",
    "TIPOFLUJO",
    "PROV_ENTREGAR",
    "PAISENTREGAR",
    "ACTIVIDAD",
]

colunas_existentes = [
    coluna
    for coluna in colunas_finais
    if coluna in df.columns
]

df_final = (
    df[colunas_existentes]
    .copy()
)

df_final.head()


,REFERENCIA,referencia_chave,FENTREGA,FCARGA,CODEUT,capacidade_norm,paletes_codeut,paletes_ocupacao_codeut,capacidade_real,ocupacao_pct,...,LOCDES,LUGARDESCARGA,TRACTORA,CODEDT,CODACT,TIPOCLIENTE,TIPOFLUJO,PROV_ENTREGAR,PAISENTREGAR,ACTIVIDAD
0,5021950798 414190635,5021950798,2026-02-02,2026-02-01,3365589,33.0,25.0,25.0,33.0,75.757576,...,319558,3. MALAQUIAS - CASH & CARRY O. AZ,93-QI-75,25531441,11,TLD PORTUGAL,Directo,Aveiro,PORTUGAL,DANONE PORTUGAL
1,2021792046,2021792046,2026-02-02,2026-02-01,3365589,33.0,25.0,25.0,33.0,75.757576,...,298515,DL Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,25532252,757,<NA>,Directo,Coimbra,PORTUGAL,SUMOLCOMPAL MARKETING
2,5021982094 414203894,5021982094,2026-02-02,2026-02-01,3365589,33.0,25.0,25.0,33.0,75.757576,...,319783,"3. MARABUTO-PRODUT.ALIMENTARES,SA",93-QI-75,25532753,11,TLD PORTUGAL,Directo,Aveiro,PORTUGAL,DANONE PORTUGAL
3,5021963392 414182550,5021963392,2026-02-02,2026-02-01,3365589,33.0,25.0,25.0,33.0,75.757576,...,319201,3. COOPERATIVA AGRICOLA DA TOCHA,93-QI-75,25534995,11,TLD PORTUGAL,Directo,Coimbra,PORTUGAL,DANONE PORTUGAL
4,5021973560 414166769,5021973560,2026-02-02,2026-01-30,3365590,33.0,35.0,35.0,33.0,106.060606,...,328547,DL TRANSPARENTODISSEIA - Transportes Unipessoa...,0000XXX,25515326,11,TLD PORTUGAL,Directo,Faro,PORTUGAL,DANONE PORTUGAL


In [44]:

# ==========================
# 1. Validar Danone no output
# ==========================
validacao = df.loc[
    df["mes"].eq(9)
].groupby("ACTIVIDAD", dropna=False).agg(
    ingresso_danone=("ingresso_danone", "sum"),
    ingresso_total=("total_ingresso", "sum"),
).reset_index()

display(
    validacao.loc[
        validacao["ACTIVIDAD"].astype("string").str.contains(
            "DANONE", case=False, na=False
        )
    ]
)


,ACTIVIDAD,ingresso_danone,ingresso_total
11,DANONE PORTUGAL,155703.986869,155703.986869
12,DANONE PORTUGAL CAPILAR,0.000000,93000.0
13,DANONE PORTUGAL TLD,0.000000,2697.43


In [45]:
# ==========================
# 1. Exportar detalhe
# ==========================
df_final.to_parquet(
    PATHS["parquet"],
    index=False,
)

# ==========================
# 2. Exportar viagens
# ==========================
df_viagens.to_parquet(
    PATHS["viagens"],
    index=False,
)

# ==========================
# 3. Exportar validações
# ==========================
validacao_financeira.to_csv(
    PATHS["validacao"],
    index=False,
    sep=";",
    decimal=",",
)

validacao_danone.to_csv(
    PATHS["danone_diag"],
    index=False,
    sep=";",
    decimal=",",
)

# ==========================
# 4. Resultado
# ==========================
pd.DataFrame({
    "output": [
        "Detalhe Power BI",
        "Viagens",
    ],
    "ficheiro": [
        str(PATHS["parquet"]),
        str(PATHS["viagens"]),
    ],
    "linhas": [
        len(df_final),
        len(df_viagens),
    ],
})

,output,ficheiro,linhas
0,Detalhe Power BI,/Users/rr/Library/Mobile Documents/com~apple~C...,132215
1,Viagens,/Users/rr/Library/Mobile Documents/com~apple~C...,19970


# 12. Análises

A partir deste ponto não são alteradas regras de negócio nem valores do pipeline.  
As análises usam apenas `df_final`, `df_viagens` e as tabelas de validação já produzidas.

In [46]:
# ==========================
# 1. Resumo financeiro mensal
# ==========================
resumo_mensal = (
    df_final
    .groupby(
        "mes_estrutura",
        as_index=False,
    )
    .agg(
        linhas=("CODEUT", "size"),
        viagens=("CODEUT", "nunique"),
        paletes=("PALETS", "sum"),
        peso_kg=("PESO_BRUTO", "sum"),
        ingresso=(
            "total_ingresso_c",
            "sum",
        ),
        custo=(
            "custo_total_c",
            "sum",
        ),
        margem=("margem", "sum"),
        estrutura=(
            "custo_estrutura",
            "sum",
        ),
        combustivel=(
            "acerto_combustivel",
            "sum",
        ),
    )
)

resumo_mensal["peso_ton"] = (
    resumo_mensal["peso_kg"]
    / 1000
)

resumo_mensal["margem_pct"] = (
    dividir_seguro(
        resumo_mensal["margem"],
        resumo_mensal["ingresso"],
    )
    * 100
)

resumo_mensal

,mes_estrutura,linhas,viagens,paletes,peso_kg,ingresso,custo,margem,estrutura,combustivel,peso_ton,margem_pct
0,2026-01-01,11327,1923,46298.001,10473868.763,327120.7235,513737.91,-186617.1865,103000.0,0.000000,10473.868763,-57.048415
1,2026-02-01,10645,1718,38238.75,8952691.761,283463.15889,458867.4,-175404.24111,103000.0,2058.953390,8952.691761,-61.879026
2,2026-03-01,12281,1966,46487.563,12553858.055,361568.980706,540931.2,-179362.219294,103000.0,2926.743206,12553.858055,-49.606639
3,2026-04-01,12577,1972,48077.75,11656719.178,400222.722927,572365.89,-172143.167073,103000.0,20632.548427,11656.719178,-43.011842
4,2026-05-01,12581,1953,47682.05,11801330.019,401786.425876,573870.16,-172083.734124,103000.0,30278.589876,11801.330019,-42.829654
5,2026-06-01,12460,1966,45447.16,11952113.721,380567.972446,580716.09,-200148.117554,103000.0,26496.304446,11952.113721,-52.59195
6,2026-07-01,14179,2167,49278.5,13429174.515,423905.369376,643897.92,-219992.550624,103000.0,21443.142376,13429.174515,-51.896618
7,2026-08-01,13273,2019,49250.0,12551971.754,597166.638806,607085.81,-9919.171194,103000.0,35006.562099,12551.971754,-1.661039
8,2026-09-01,15856,2090,50019.0,12413699.8,679647.17005,640188.04,39459.13005,103000.0,46703.436681,12413.6998,5.805826
9,2026-10-01,16618,2095,49841.18,12526923.737,693045.80462,661924.34,31121.46462,103000.0,57416.702087,12526.923737,4.490535


In [47]:
# ==========================
# 1. KPIs de viagens
# ==========================
viagens_validas = (
    df_viagens.loc[
        df_viagens[
            "ocupacao_estado"
        ].eq("OK")
    ]
    .copy()
)

kpi_viagens = pd.DataFrame({
    "metrica": [
        "Viagens",
        "Paletes",
        "Peso ton",
        "Ocupação média %",
        "Custo por viagem",
        "Custo por palete",
        "Custo por ton",
        "Margem",
    ],
    "valor": [
        df_viagens["CODEUT"].nunique(),
        df_viagens["paletes"].sum(),
        df_viagens["peso_ton"].sum(),
        viagens_validas[
            "ocupacao_pct"
        ].mean(),
        df_viagens[
            "custo_total"
        ].mean(),
        dividir_seguro(
            pd.Series([
                df_viagens[
                    "custo_total"
                ].sum()
            ]),
            pd.Series([
                df_viagens[
                    "paletes"
                ].sum()
            ]),
        ).iloc[0],
        dividir_seguro(
            pd.Series([
                df_viagens[
                    "custo_total"
                ].sum()
            ]),
            pd.Series([
                df_viagens[
                    "peso_ton"
                ].sum()
            ]),
        ).iloc[0],
        df_viagens["margem"].sum(),
    ],
})

kpi_viagens

,metrica,valor
0,Viagens,1.997000e+04
1,Paletes,4.714380e+05
2,Peso ton,1.185225e+05
3,Ocupação média %,7.992895e+01
4,Custo por viagem,2.960076e+02
5,Custo por palete,1.253881e+01
6,Custo por ton,4.987466e+01
7,Margem,-1.348958e+06


In [48]:

# ==========================
# 1. Viagens únicas
# ==========================

validacao_enchimento = (
    df.loc[
        df["dados_validos"]
        & df["TRANSPORTISTA"].notna()
    ]
    .groupby(
        "CODEUT",
        as_index=False,
    )
    .agg(
        transportador=(
            "TRANSPORTISTA",
            "first",
        ),
        paletes=(
            "paletes_codeut",
            "first",
        ),
        paletes_ocupacao=(
            "paletes_ocupacao_codeut",
            "first",
        ),
        capacidade=(
            "capacidade_norm",
            "first",
        ),
    )
)


# ==========================
# 2. Taxa por viagem
# ==========================

validacao_enchimento["taxa_enchimento"] = (
    validacao_enchimento["paletes_ocupacao"]
    / validacao_enchimento["capacidade"]
)

validacao_enchimento["taxa_enchimento_pct"] = (
    validacao_enchimento["taxa_enchimento"]
    * 100
)


# ==========================
# 3. Resumo por transportador
# ==========================

enchimento_transportador = (
    validacao_enchimento
    .groupby(
        "transportador",
        as_index=False,
    )
    .agg(
        viagens=(
            "CODEUT",
            "nunique",
        ),
        paletes=(
            "paletes_ocupacao",
            "sum",
        ),
        capacidade_total=(
            "capacidade",
            "sum",
        ),
        taxa_media_viagem_pct=(
            "taxa_enchimento_pct",
            "mean",
        ),
    )
)


# ==========================
# 4. Taxa ponderada
# ==========================

enchimento_transportador[
    "taxa_enchimento_pct"
] = (
    enchimento_transportador["paletes"]
    / enchimento_transportador["capacidade_total"]
    * 100
)


# ==========================
# 5. Formatar
# ==========================

enchimento_transportador = (
    enchimento_transportador
    .sort_values(
        "taxa_enchimento_pct",
        ascending=False,
    )
    .reset_index(drop=True)
)

enchimento_transportador[
    [
        "taxa_media_viagem_pct",
        "taxa_enchimento_pct",
    ]
] = (
    enchimento_transportador[
        [
            "taxa_media_viagem_pct",
            "taxa_enchimento_pct",
        ]
    ]
    .round(2)
)


# ==========================
# 6. Resultado
# ==========================

enchimento_transportador

,transportador,viagens,paletes,capacidade_total,taxa_media_viagem_pct,taxa_enchimento_pct
0,NO USAR,9,297.0,297.0,100.0,100.0
1,MOSCA PORTUGAL LDA,64,2112.0,2112.0,100.0,100.0
2,EL MOSCA S.A.,4,132.0,132.0,100.0,100.0
3,LOGISTA FREIGHT S.A.U.,41,1353.0,1353.0,100.0,100.0
4,SALVESEN,223,12411.25,12780.0,99.76,97.11
5,TORRESTIR - TRANSPORTES NACIONAIS E INTERNACIO...,180,8525.0,8987.0,99.51,94.86
6,TRANSPORTES CARGAKER LDA,27,1239.0,1408.0,85.15,88.0
7,JMR PRESTAÇAO DE SERVIÇOS PARA A DISTRIBUÇAO S.A,2071,69946.0,81493.0,85.8,85.83
8,TRANSPORTES PAULO COSTA & FERREIRA LDA,493,15876.0,18758.0,84.49,84.64
9,MODELO CONTINENTE HIPERMERCADOS S.A.,1396,42428.73,50420.0,83.82,84.15


In [49]:
# ==========================
# 7. Média global
# ==========================

taxa_enchimento_global_pct = (
    validacao_enchimento["paletes_ocupacao"].sum()
    / validacao_enchimento["capacidade"].sum()
    * 100
)

taxa_enchimento_global_pct

np.float64(76.81635155663072)

In [50]:

# ==========================
# 1. Consultar origem
# ==========================
import duckdb

with duckdb.connect(str(PATHS["db"]), read_only=True) as con:
    df_validacao_origem = con.execute("""
        WITH origem AS (
            SELECT
                SUBSTR(TRIM(CAST(FENTREGA AS VARCHAR)), 1, 6) AS MES,
                TRY_CAST(CODACT AS INTEGER) AS CODACT,
                NULLIF(TRIM(CAST(GESTION AS VARCHAR)), '') AS GESTION,
                TRIM(CAST(PROPIETARIO AS VARCHAR)) AS PROPIETARIO,
                NULLIF(TRIM(CAST(CODEUT AS VARCHAR)), '') AS CODEUT,

                TRY_CAST(
                    REPLACE(TRIM(CAST(INGRESODT AS VARCHAR)), ',', '.')
                    AS DECIMAL(18, 2)
                ) AS INGRESODT,

                TRY_CAST(
                    REPLACE(TRIM(CAST(COSTEDT AS VARCHAR)), ',', '.')
                    AS DECIMAL(18, 2)
                ) AS COSTEDT,

                TRY_CAST(
                    REPLACE(TRIM(CAST(PALETS AS VARCHAR)), ',', '.')
                    AS DOUBLE
                ) AS PALETS,

                TRY_CAST(
                    REPLACE(TRIM(CAST(PESO_BRUTO AS VARCHAR)), ',', '.')
                    AS DOUBLE
                ) AS PESO_BRUTO

            FROM inform_27_2026
            WHERE SUBSTR(TRIM(CAST(FENTREGA AS VARCHAR)), 1, 6)
                  IN ('202607', '202608', '202609', '202610')
        ),

        classificada AS (
            SELECT *,
                CODACT IS NOT NULL
                AND CODACT <> 13
                AND (
                    GESTION = 'LIS'
                    OR (
                        GESTION IS NULL
                        AND PROPIETARIO IN ('CEP', 'PTG')
                    )
                ) AS FINANCEIRO
            FROM origem
        )

        SELECT
            MES,
            COUNT(*) AS LINHAS_RAW,
            COUNT(*) FILTER (WHERE FINANCEIRO) AS LINHAS_FINANCEIRO,
            COUNT(DISTINCT CODEUT) FILTER (WHERE FINANCEIRO) AS VIAGENS,

            ROUND(
                SUM(PALETS) FILTER (WHERE FINANCEIRO), 2
            ) AS PALETES,

            ROUND(
                SUM(PESO_BRUTO) FILTER (WHERE FINANCEIRO), 2
            ) AS PESO_KG,

            ROUND(SUM(INGRESODT), 2) AS INGRESSO_RAW,
            ROUND(SUM(COSTEDT), 2) AS CUSTO_RAW,

            ROUND(
                SUM(INGRESODT) FILTER (WHERE FINANCEIRO), 2
            ) AS INGRESSO_FINANCEIRO,

            ROUND(
                SUM(COSTEDT) FILTER (WHERE FINANCEIRO), 2
            ) AS CUSTO_FINANCEIRO

        FROM classificada
        GROUP BY MES
        ORDER BY MES
    """).df()

# ==========================
# 2. Resultado
# ==========================
df_validacao_origem


,MES,LINHAS_RAW,LINHAS_FINANCEIRO,VIAGENS,PALETES,PESO_KG,INGRESSO_RAW,CUSTO_RAW,INGRESSO_FINANCEIRO,CUSTO_FINANCEIRO
0,202607,100348,14127,2167,49017.50,13299174.52,3671504.30,8539053.70,406394.21,536690.18
1,202608,90440,13227,2020,49028.00,12434471.75,3173223.21,6567442.12,446345.62,499971.34
2,202609,15856,12713,1978,46723.00,12150463.39,435003.37,537188.04,435003.37,473748.50
3,202610,16618,13468,1989,46507.18,12285313.09,441068.86,558924.34,441068.86,497931.57


In [51]:
# ==========================
# 1. Ingressos Danone por actividade
# ==========================
mask_periodo = (
    df_final["ano"].eq(ANO)
    & df_final["mes"].isin([8, 9, 10])
    & df_final["CODACT"].isin([11, 13])
)

resumo_danone_final = (
    df_final.loc[mask_periodo]
    .assign(
        DANONE_011=lambda x: x["ingresso_danone"].where(x["CODACT"].eq(11), 0),
        CAPILAR_013=lambda x: x["ingresso_capilar"].where(x["CODACT"].eq(13), 0),
        COMBUSTIVEL=lambda x: x["acerto_combustivel"],
    )
    .groupby("mes", as_index=False)
    .agg(
        DANONE_011=("DANONE_011", "sum"),
        CAPILAR_013=("CAPILAR_013", "sum"),
        COMBUSTIVEL=("COMBUSTIVEL", "sum"),
        TOTAL_COM_COMBUSTIVEL=("total_ingresso_c", "sum"),
    )
)
resumo_danone_final["TOTAL_SEM_COMBUSTIVEL"] = (
    resumo_danone_final["DANONE_011"]
    + resumo_danone_final["CAPILAR_013"]
)
resumo_danone_final["MES"] = resumo_danone_final["mes"].map(
    {8: "Agosto", 9: "Setembro", 10: "Outubro"}
)
resumo_danone_final = resumo_danone_final[
    ["MES", "DANONE_011", "CAPILAR_013", "TOTAL_SEM_COMBUSTIVEL", "COMBUSTIVEL", "TOTAL_COM_COMBUSTIVEL"]
].round(2)

# ==========================
# 2. Controlar valores fixos
# ==========================
for mes, esperado in INGRESSO_CAPILAR_MENSAL.items():
    obtido = df_final.loc[
        df_final["ano"].eq(ANO) & df_final["mes"].eq(mes),
        "ingresso_capilar",
    ].sum()
    if abs(obtido - esperado) > TOLERANCIA_FINANCEIRA:
        raise ValueError(f"CAPILAR {mes:02d}: {obtido:.2f} € != {esperado:.2f} €")

# ==========================
# 3. Resultado
# ==========================
resumo_danone_final


,MES,DANONE_011,CAPILAR_013,TOTAL_SEM_COMBUSTIVEL,COMBUSTIVEL,TOTAL_COM_COMBUSTIVEL
0,Agosto,161721.48,0.0,161721.48,10754.48,172475.96
1,Setembro,155703.99,93000.0,248703.99,15570.40,264274.39
2,Outubro,159779.43,85000.0,244779.43,19636.89,264416.32


In [52]:
# ==========================
# 1. Resumo mensal de todos os clientes
# ==========================
import pandas as pd

meses = [8, 9, 10]
meses_nomes = {8: "Agosto", 9: "Setembro", 10: "Outubro"}
periodo = df_final.loc[
    df_final["ano"].eq(ANO) & df_final["mes"].isin(meses)
].copy()

periodo["DANONE_TARIFA"] = periodo["ingresso_danone_base"].where(
    periodo["CODACT"].eq(11), 0.0
)
periodo["DANONE_EXTRA"] = periodo["extra_danone"].where(
    periodo["CODACT"].eq(11), 0.0
)
periodo["DANONE_FINAL"] = periodo["ingresso_danone"].where(
    periodo["CODACT"].eq(11), 0.0
)
periodo["CAPILAR_013"] = periodo["ingresso_capilar"].where(
    periodo["CODACT"].eq(13), 0.0
)

resumo_ingressos_final = (
    periodo.groupby("mes", as_index=False).agg(
        DANONE_TARIFA=("DANONE_TARIFA", "sum"),
        DANONE_EXTRA=("DANONE_EXTRA", "sum"),
        DANONE_FINAL=("DANONE_FINAL", "sum"),
        CAPILAR_013=("CAPILAR_013", "sum"),
        OUTROS_CLIENTES_BASE=("INGRESODT", "sum"),
        INGRESSO_TOTAL_SEM_COMBUSTIVEL=("total_ingresso", "sum"),
        COMBUSTIVEL_TODOS_CLIENTES=("acerto_combustivel", "sum"),
        INGRESSO_TOTAL_TODOS_CLIENTES=("total_ingresso_c", "sum"),
    )
    .set_index("mes")
    .reindex(meses, fill_value=0.0)
    .reset_index()
)
resumo_ingressos_final["MES"] = resumo_ingressos_final["mes"].map(meses_nomes)
resumo_ingressos_final = resumo_ingressos_final.drop(columns="mes")
resumo_ingressos_final = resumo_ingressos_final[[
    "MES", "DANONE_TARIFA", "DANONE_EXTRA", "DANONE_FINAL",
    "CAPILAR_013", "OUTROS_CLIENTES_BASE",
    "INGRESSO_TOTAL_SEM_COMBUSTIVEL", "COMBUSTIVEL_TODOS_CLIENTES",
    "INGRESSO_TOTAL_TODOS_CLIENTES",
]]

# ==========================
# 2. Reconciliar Danone CSV e associação
# ==========================
diagnostico_danone = danone_chave.loc[
    danone_chave["PFEENT"].dt.year.eq(ANO)
    & danone_chave["PFEENT"].dt.month.isin(meses)
].copy()
diagnostico_danone["mes"] = diagnostico_danone["PFEENT"].dt.month
diagnostico_danone["BASE_ASSOCIADA"] = diagnostico_danone[
    "ingresso_danone_base_total"
].where(diagnostico_danone["ASSOCIADO"], 0.0)
diagnostico_danone["BASE_POR_ASSOCIAR"] = diagnostico_danone[
    "ingresso_danone_base_total"
].where(~diagnostico_danone["ASSOCIADO"], 0.0)
diagnostico_danone["ENTREGAS_POR_ASSOCIAR"] = (
    ~diagnostico_danone["ASSOCIADO"]
    & diagnostico_danone["ingresso_danone_base_total"].notna()
).astype(int)
reconciliacao_danone_final = (
    diagnostico_danone.groupby("mes", as_index=False)
    .agg(
        BASE_CSV=("ingresso_danone_base_total", "sum"),
        BASE_ASSOCIADA=("BASE_ASSOCIADA", "sum"),
        BASE_POR_ASSOCIAR=("BASE_POR_ASSOCIAR", "sum"),
        ENTREGAS_POR_ASSOCIAR=("ENTREGAS_POR_ASSOCIAR", "sum"),
    )
    .set_index("mes").reindex(meses, fill_value=0.0).reset_index()
)
reconciliacao_danone_final["MES"] = reconciliacao_danone_final["mes"].map(meses_nomes)
reconciliacao_danone_final = reconciliacao_danone_final[
    ["MES", "BASE_CSV", "BASE_ASSOCIADA", "BASE_POR_ASSOCIAR", "ENTREGAS_POR_ASSOCIAR"]
]

# ==========================
# 3. Controlo financeiro
# ==========================
for mes in meses:
    fatia = periodo.loc[periodo["mes"].eq(mes)]
    extra = fatia.loc[fatia["CODACT"].eq(11), "extra_danone"].sum()
    if fatia.loc[fatia["CODACT"].eq(11), "ingresso_danone_base"].gt(0).any():
        if abs(extra - EXTRA_DANONE_MENSAL) > TOLERANCIA_FINANCEIRA:
            raise ValueError(f"Danone {mes:02d}: extra {extra:.2f} != {EXTRA_DANONE_MENSAL:.2f} €")
    diferenca = (
        fatia["INGRESODT"].fillna(0.0).sum()
        + fatia["ingresso_danone"].fillna(0.0).sum()
        + fatia["ingresso_capilar"].fillna(0.0).sum()
        + fatia["acerto_combustivel"].fillna(0.0).sum()
        - fatia["total_ingresso_c"].sum()
    )
    if abs(diferenca) > TOLERANCIA_FINANCEIRA:
        raise ValueError(f"Componentes do ingresso total não conciliam em {mes:02d}")
    total = fatia["total_ingresso_c"].sum()
    resumo_total = resumo_ingressos_final.loc[
        resumo_ingressos_final["MES"].eq(meses_nomes[mes]),
        "INGRESSO_TOTAL_TODOS_CLIENTES",
    ].iloc[0]
    if abs(total - resumo_total) > TOLERANCIA_FINANCEIRA:
        raise ValueError(f"Ingresso total inconsistente em {mes:02d}")

# ==========================
# 4. Resultados
# ==========================
display(resumo_ingressos_final.round(2))
display(reconciliacao_danone_final.round(2))


,MES,DANONE_TARIFA,DANONE_EXTRA,DANONE_FINAL,CAPILAR_013,OUTROS_CLIENTES_BASE,INGRESSO_TOTAL_SEM_COMBUSTIVEL,COMBUSTIVEL_TODOS_CLIENTES,INGRESSO_TOTAL_TODOS_CLIENTES
0,Agosto,141721.48,20000.0,161721.48,0.0,400438.6,562160.08,35006.56,597166.64
1,Setembro,135703.99,20000.0,155703.99,93000.0,384239.75,632943.73,46703.44,679647.17
2,Outubro,139779.43,20000.0,159779.43,85000.0,390849.67,635629.1,57416.70,693045.8


,MES,BASE_CSV,BASE_ASSOCIADA,BASE_POR_ASSOCIAR,ENTREGAS_POR_ASSOCIAR
0,Agosto,149023.33,141721.48,7301.85,1802
1,Setembro,146259.36,135703.99,10555.37,1955
2,Outubro,146788.7,139779.43,7009.27,1804


In [53]:

# ==========================
# 1. Filtrar dados
# ==========================
import pandas as pd
from IPython.display import display

df_transportes = df_final.loc[
    pd.to_numeric(df_final["ano"], errors="coerce").eq(2026)
    & ~pd.to_numeric(df_final["CODACT"], errors="coerce").eq(13)
    & df_final["CODEUT"].notna()
].copy()

df_transportes["mes"] = pd.to_numeric(
    df_transportes["mes"], errors="coerce"
).astype("Int64")

for coluna in ["PALETS", "COSTEDT"]:
    df_transportes[coluna] = pd.to_numeric(
        df_transportes[coluna], errors="coerce"
    ).fillna(0)

# ==========================
# 2. Agregar por mês
# ==========================
df_resumo = (
    df_transportes
    .groupby("mes")
    .agg(
        VIAGENS=("CODEUT", "nunique"),
        PALETES=("PALETS", "sum"),
        CUSTO=("COSTEDT", "sum")
    )
    .reindex(range(1, 11), fill_value=0)
)

df_resumo["MEDIA_PALETES_ROTA"] = (
    df_resumo["PALETES"]
    / df_resumo["VIAGENS"].replace(0, float("nan"))
)

df_resumo["CUSTO_PALETE"] = (
    df_resumo["CUSTO"]
    / df_resumo["PALETES"].replace(0, float("nan"))
)

# ==========================
# 3. Tabela mensal
# ==========================
meses = {
    1: "Jan", 2: "Fev", 3: "Mar",
    4: "Abr", 5: "Mai", 6: "Jun",
    7: "Jul", 8: "Ago", 9: "Set",
    10: "Out"
}

df_tabela = df_resumo[
    ["VIAGENS", "PALETES", "MEDIA_PALETES_ROTA", "CUSTO_PALETE"]
].T

df_tabela.index = [
    "Nº Viagens",
    "Nº Paletes",
    "Média paletes/rota",
    "Custo/Palete (€)"
]

df_tabela.columns = [meses[m] for m in df_tabela.columns]

# ==========================
# 4. Comparação mensal
# ==========================
df_comparacao = pd.DataFrame({
    "Métrica": df_tabela.index,
    "Setembro": df_tabela["Set"].values,
    "Outubro": df_tabela["Out"].values
})

df_comparacao["Variação %"] = (
    (df_comparacao["Outubro"] / df_comparacao["Setembro"] - 1) * 100
).where(df_comparacao["Setembro"] != 0)

# ==========================
# 5. Resultados
# ==========================
display(df_tabela.round(2))
display(df_comparacao.round(2))


,Jan,Fev,Mar,Abr,Mai,Jun,Jul,Ago,Set,Out
Nº Viagens,1916,1710,1958,1970,1949,1965,2167,2018,1978,1989
Nº Paletes,46185.001,38165.0,46236.563,47842.25,47437.05,45258.16,49017.5,49028.0,46723.0,46507.18
Média paletes/rota,24.104907,22.318713,23.614179,24.285406,24.339174,23.032142,22.619982,24.295342,23.621335,23.382192
Custo/Palete (€),8.821331,9.248984,9.360931,9.709198,9.814694,10.461567,10.94895,10.197669,10.139514,10.706553


,Métrica,Setembro,Outubro,Variação %
0,Nº Viagens,1978,1989,0.556117
1,Nº Paletes,46723.0,46507.18,-0.461914
2,Média paletes/rota,23.621335,23.382192,-1.012401
3,Custo/Palete (€),10.139514,10.706553,5.592368


In [55]:

# ==========================
# 1. Bibliotecas
# ==========================
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
from pptx.enum.shapes import MSO_SHAPE
from pathlib import Path
import pandas as pd


# ==========================
# 2. Configuração do ficheiro
# ==========================
from pathlib import Path

PASTA_OUTPUT = Path.home() / "Downloads"
PASTA_OUTPUT.mkdir(parents=True, exist_ok=True)

OUTPUT = PASTA_OUTPUT / "Transportes_Setembro_Outubro_2026.pptx"

AZUL = RGBColor(21, 43, 77)
VERDE = RGBColor(0, 132, 145)
BRANCO = RGBColor(255, 255, 255)
CINZA = RGBColor(243, 244, 245)
PRETO = RGBColor(35, 35, 35)
VERMELHO = RGBColor(210, 45, 45)
VERDE_POS = RGBColor(28, 125, 65)

prs = Presentation()
prs.slide_width = Inches(13.333)
prs.slide_height = Inches(7.5)

slide = prs.slides.add_slide(prs.slide_layouts[6])

# ==========================
# 3. Funções auxiliares
# ==========================
def texto(x, y, w, h, valor, tamanho=12, cor=PRETO,
          negrito=False, alinhamento=PP_ALIGN.LEFT):

    caixa = slide.shapes.add_textbox(
        Inches(x), Inches(y), Inches(w), Inches(h)
    )
    tf = caixa.text_frame
    tf.word_wrap = True
    tf.margin_left = Inches(0.04)
    tf.margin_right = Inches(0.04)
    tf.margin_top = Inches(0.02)
    tf.margin_bottom = Inches(0.02)

    p = tf.paragraphs[0]
    p.alignment = alinhamento

    run = p.add_run()
    run.text = str(valor)
    run.font.name = "Arial"
    run.font.size = Pt(tamanho)
    run.font.bold = negrito
    run.font.color.rgb = cor

    return caixa


def tabela(x, y, larguras, altura_linha, linhas,
           colunas_destaque=None, primeira_coluna=True):

    colunas_destaque = colunas_destaque or []
    largura_total = sum(larguras)
    altura_total = altura_linha * len(linhas)

    shape = slide.shapes.add_table(
        len(linhas), len(larguras),
        Inches(x), Inches(y),
        Inches(largura_total), Inches(altura_total)
    )

    tb = shape.table

    for j, largura in enumerate(larguras):
        tb.columns[j].width = Inches(largura)

    for i, linha in enumerate(linhas):
        tb.rows[i].height = Inches(altura_linha)

        for j, valor in enumerate(linha):
            cell = tb.cell(i, j)
            cell.text = str(valor)

            if i == 0:
                cell.fill.solid()
                cell.fill.fore_color.rgb = VERDE
            elif j in colunas_destaque:
                cell.fill.solid()
                cell.fill.fore_color.rgb = RGBColor(225, 244, 246)
            elif i % 2 == 0:
                cell.fill.solid()
                cell.fill.fore_color.rgb = CINZA
            else:
                cell.fill.solid()
                cell.fill.fore_color.rgb = BRANCO

            cell.margin_left = Inches(0.04)
            cell.margin_right = Inches(0.04)
            cell.margin_top = Inches(0.02)
            cell.margin_bottom = Inches(0.02)
            cell.vertical_anchor = MSO_ANCHOR.MIDDLE

            for p in cell.text_frame.paragraphs:
                p.alignment = (
                    PP_ALIGN.LEFT
                    if j == 0 else PP_ALIGN.CENTER
                )

                for run in p.runs:
                    run.font.name = "Arial"
                    run.font.size = Pt(9 if j == 0 else 8)
                    run.font.bold = i == 0 or (
                        primeira_coluna and j == 0
                    )
                    run.font.color.rgb = (
                        BRANCO if i == 0 else PRETO
                    )

    return tb


# ==========================
# 4. Preparar tabela mensal
# ==========================
meses = [
    "Jan", "Fev", "Mar", "Abr", "Mai",
    "Jun", "Jul", "Ago", "Set", "Out"
]

metricas = [
    "Nº Viagens",
    "Nº Paletes",
    "Média paletes/rota",
    "Custo/Palete (€)"
]

def formatar_valor(valor, metrica):
    if pd.isna(valor):
        return "—"

    if metrica in ["Nº Viagens", "Nº Paletes"]:
        return f"{valor:,.0f}".replace(",", " ")

    if metrica == "Custo/Palete (€)":
        return f"{valor:.2f} €".replace(".", ",")

    return f"{valor:.1f}".replace(".", ",")


linhas_mensais = [["2026"] + meses]

for metrica in metricas:
    linhas_mensais.append(
        [metrica] + [
            formatar_valor(df_tabela.loc[metrica, mes], metrica)
            for mes in meses
        ]
    )

# ==========================
# 5. Preparar comparação
# ==========================
linhas_comparacao = [
    ["Métrica", "Set/2026", "Out/2026", "Δ %"]
]

for _, linha in df_comparacao.iterrows():
    metrica = linha["Métrica"]
    variacao = linha["Variação %"]

    linhas_comparacao.append([
        metrica,
        formatar_valor(linha["Setembro"], metrica),
        formatar_valor(linha["Outubro"], metrica),
        (
            f"{variacao:+.1f}%".replace(".", ",")
            if pd.notna(variacao) else "—"
        )
    ])

# ==========================
# 6. Cabeçalho
# ==========================
texto(
    0.55, 0.38, 8, 0.5,
    "5. TRANSPORTES",
    23, AZUL, True
)

texto(
    0.55, 0.93, 9, 0.3,
    "Evolução Operacional — 2026",
    13, RGBColor(240, 95, 20), True
)

# ==========================
# 7. Tabelas
# ==========================
tabela(
    0.50, 1.65,
    [1.45] + [0.68] * 10,
    0.49,
    linhas_mensais,
    colunas_destaque=[9, 10]
)

tabela(
    9.00, 1.65,
    [1.50, 0.90, 0.90, 0.78],
    0.49,
    linhas_comparacao
)

# ==========================
# 8. Destaque Setembro e Outubro
# ==========================
destaque = slide.shapes.add_shape(
    MSO_SHAPE.ROUNDED_RECTANGLE,
    Inches(6.62), Inches(1.60),
    Inches(1.42), Inches(2.55)
)

destaque.fill.background()
destaque.line.color.rgb = RGBColor(235, 65, 65)
destaque.line.width = Pt(1.5)

# ==========================
# 9. Nota e rodapé
# ==========================
texto(
    9.05, 4.45, 3.75, 0.8,
    "Comparação Setembro vs Outubro de 2026.\n"
    "Universo: todas as actividades excepto CODACT 013.",
    10, PRETO
)

texto(
    0.55, 6.95, 4.5, 0.3,
    "SALVESEN LOGÍSTICA  |  TRANSPORTES",
    11, VERDE, True
)

texto(
    8.30, 7.00, 4.40, 0.2,
    "CONFIDENTIAL AND PROPRIETARY MATERIAL",
    7, RGBColor(125, 125, 125),
    alinhamento=PP_ALIGN.RIGHT
)

# ==========================
# 10. Exportar PowerPoint
# ==========================
prs.save(OUTPUT)

print(f"PowerPoint criado: {OUTPUT}")


PowerPoint criado: /Users/rr/Downloads/Transportes_Setembro_Outubro_2026.pptx


In [60]:

# ==========================
# 1. Consultar DuckDB
# ==========================
import duckdb
import pandas as pd
from IPython.display import display

with duckdb.connect(str(PATHS["db"]), read_only=True) as con:
    df_capilar_origem = con.execute("""
        WITH base AS (
            SELECT
                SUBSTR(TRIM(CAST(FENTREGA AS VARCHAR)), 1, 6) AS MES,
                NULLIF(TRIM(CAST(CODEUT AS VARCHAR)), '') AS CODEUT,

                TRY_CAST(
                    REPLACE(TRIM(CAST(PALETS AS VARCHAR)), ',', '.')
                    AS DOUBLE
                ) AS PALETES,

                TRY_CAST(
                    REPLACE(TRIM(CAST(COSTEDT AS VARCHAR)), ',', '.')
                    AS DOUBLE
                ) AS CUSTO

            FROM inform_27_2026

            WHERE TRY_CAST(CODACT AS INTEGER) = 13
              AND SUBSTR(TRIM(CAST(FENTREGA AS VARCHAR)), 1, 6)
                  BETWEEN '202601' AND '202610'
        )

        SELECT
            MES,
            COUNT(*) AS REGISTOS,
            COUNT(DISTINCT CODEUT) AS VIAGENS,
            ROUND(SUM(PALETES), 2) AS PALETES,
            ROUND(SUM(CUSTO), 2) AS CUSTO

        FROM base
        GROUP BY MES
        ORDER BY MES
    """).df()

# ==========================
# 2. Calcular indicadores
# ==========================
df_capilar_origem["MEDIA_PALETES_ROTA"] = (
    df_capilar_origem["PALETES"]
    / df_capilar_origem["VIAGENS"].replace(0, float("nan"))
)

df_capilar_origem["CUSTO_PALETE"] = (
    df_capilar_origem["CUSTO"]
    / df_capilar_origem["PALETES"].replace(0, float("nan"))
)

# ==========================
# 3. Resultado
# ==========================
df_capilar_origem


,MES,REGISTOS,VIAGENS,PALETES,CUSTO,MEDIA_PALETES_ROTA,CUSTO_PALETE
0,202601,3001,419,3681.0,61331.55,8.785203,16.661654
1,202602,2991,410,3675.0,64574.39,8.963415,17.571263
2,202603,3393,460,4207.6,73664.00,9.146957,17.507368
3,202604,3201,434,3990.0,68267.85,9.193548,17.109737
4,202605,2998,393,3609.0,64196.88,9.183206,17.787997
5,202606,3076,392,3275.0,63182.74,8.354592,19.292440
6,202607,3435,427,3459.0,66242.04,8.100703,19.150633
7,202608,3037,380,3046.0,56853.64,8.015789,18.665016
8,202609,3102,406,3118.0,60067.70,7.679803,19.264817
9,202610,3100,396,3118.0,57443.50,7.873737,18.423188


In [62]:

# ==========================
# 1. Bibliotecas
# ==========================
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
from pptx.enum.shapes import MSO_SHAPE
from pathlib import Path
import pandas as pd

# ==========================
# 2. Configuração
# ==========================
PASTA_OUTPUT = Path.home() / "Downloads"
PASTA_OUTPUT.mkdir(parents=True, exist_ok=True)

OUTPUT = PASTA_OUTPUT / "Capilar_Setembro_Outubro_2026.pptx"

AZUL = RGBColor(21, 43, 77)
VERDE = RGBColor(0, 132, 145)
BRANCO = RGBColor(255, 255, 255)
CINZA = RGBColor(243, 244, 245)
PRETO = RGBColor(35, 35, 35)
VERMELHO = RGBColor(210, 45, 45)

prs = Presentation()
prs.slide_width = Inches(13.333)
prs.slide_height = Inches(7.5)

slide = prs.slides.add_slide(prs.slide_layouts[6])

# ==========================
# 3. Preparar dados CAPILAR
# ==========================
df_capilar = df_capilar_origem.copy()

colunas_obrigatorias = [
    "MES",
    "VIAGENS",
    "PALETES",
    "CUSTO"
]

em_falta = [
    coluna for coluna in colunas_obrigatorias
    if coluna not in df_capilar.columns
]

if em_falta:
    raise ValueError(f"Colunas em falta: {em_falta}")

df_capilar["MES"] = pd.to_numeric(
    df_capilar["MES"], errors="coerce"
).astype("Int64")

df_capilar["MES"] = df_capilar["MES"].mod(100)

for coluna in ["VIAGENS", "PALETES", "CUSTO"]:
    df_capilar[coluna] = pd.to_numeric(
        df_capilar[coluna], errors="coerce"
    )

df_capilar = df_capilar.loc[
    df_capilar["MES"].between(1, 10)
].copy()

if df_capilar.empty:
    raise ValueError("Não existem dados CAPILAR de Janeiro a Outubro.")

if df_capilar[["VIAGENS", "PALETES", "CUSTO"]].isna().any().any():
    raise ValueError("Existem valores operacionais inválidos.")

if df_capilar["MES"].duplicated().any():
    raise ValueError("Existem meses duplicados em df_capilar_origem.")

df_capilar = (
    df_capilar
    .set_index("MES")
    .reindex(range(1, 11))
)

if df_capilar[["VIAGENS", "PALETES", "CUSTO"]].isna().any().any():
    raise ValueError("Existem meses sem dados CAPILAR.")

# ==========================
# 4. Indicadores mensais
# ==========================
df_capilar["MEDIA_PALETES_ROTA"] = (
    df_capilar["PALETES"]
    / df_capilar["VIAGENS"].replace(0, float("nan"))
)

df_capilar["CUSTO_PALETE"] = (
    df_capilar["CUSTO"]
    / df_capilar["PALETES"].replace(0, float("nan"))
)

meses = {
    1: "Jan", 2: "Fev", 3: "Mar",
    4: "Abr", 5: "Mai", 6: "Jun",
    7: "Jul", 8: "Ago", 9: "Set",
    10: "Out"
}

metricas = {
    "Nº Viagens": "VIAGENS",
    "Nº Paletes": "PALETES",
    "Média paletes/rota": "MEDIA_PALETES_ROTA",
    "Custo/Palete (€)": "CUSTO_PALETE"
}

df_tabela_capilar = df_capilar[
    list(metricas.values())
].T

df_tabela_capilar.index = list(metricas.keys())
df_tabela_capilar.columns = [
    meses[mes] for mes in df_tabela_capilar.columns
]

# ==========================
# 5. Comparação mensal
# ==========================
df_comparacao_capilar = pd.DataFrame({
    "Métrica": list(metricas.keys()),
    "Setembro": df_tabela_capilar["Set"].values,
    "Outubro": df_tabela_capilar["Out"].values
})

df_comparacao_capilar["Variação %"] = (
    (
        df_comparacao_capilar["Outubro"]
        / df_comparacao_capilar["Setembro"].replace(
            0, float("nan")
        )
        - 1
    ) * 100
)

# ==========================
# 6. Funções auxiliares
# ==========================
def texto_capilar(x, y, w, h, valor, tamanho=12,
                  cor=PRETO, negrito=False,
                  alinhamento=PP_ALIGN.LEFT):

    caixa = slide.shapes.add_textbox(
        Inches(x), Inches(y), Inches(w), Inches(h)
    )

    tf = caixa.text_frame
    tf.word_wrap = True
    tf.margin_left = Inches(0.04)
    tf.margin_right = Inches(0.04)
    tf.margin_top = Inches(0.02)
    tf.margin_bottom = Inches(0.02)

    p = tf.paragraphs[0]
    p.alignment = alinhamento

    run = p.add_run()
    run.text = str(valor)
    run.font.name = "Arial"
    run.font.size = Pt(tamanho)
    run.font.bold = negrito
    run.font.color.rgb = cor

    return caixa


def tabela_capilar(x, y, larguras, altura_linha, linhas,
                   colunas_destaque=None):

    colunas_destaque = colunas_destaque or []

    shape = slide.shapes.add_table(
        len(linhas),
        len(larguras),
        Inches(x),
        Inches(y),
        Inches(sum(larguras)),
        Inches(altura_linha * len(linhas))
    )

    tb = shape.table

    for j, largura in enumerate(larguras):
        tb.columns[j].width = Inches(largura)

    for i, linha in enumerate(linhas):
        tb.rows[i].height = Inches(altura_linha)

        for j, valor in enumerate(linha):
            cell = tb.cell(i, j)
            cell.text = str(valor)
            cell.fill.solid()

            if i == 0:
                cell.fill.fore_color.rgb = VERDE
            elif j in colunas_destaque:
                cell.fill.fore_color.rgb = RGBColor(225, 244, 246)
            elif i % 2 == 0:
                cell.fill.fore_color.rgb = CINZA
            else:
                cell.fill.fore_color.rgb = BRANCO

            cell.margin_left = Inches(0.04)
            cell.margin_right = Inches(0.04)
            cell.margin_top = Inches(0.02)
            cell.margin_bottom = Inches(0.02)
            cell.vertical_anchor = MSO_ANCHOR.MIDDLE

            for p in cell.text_frame.paragraphs:
                p.alignment = (
                    PP_ALIGN.LEFT if j == 0 else PP_ALIGN.CENTER
                )

                for run in p.runs:
                    run.font.name = "Arial"
                    run.font.size = Pt(9 if j == 0 else 8)
                    run.font.bold = i == 0 or j == 0
                    run.font.color.rgb = (
                        BRANCO if i == 0 else PRETO
                    )

    return tb


def formatar_capilar(valor, metrica):
    if pd.isna(valor):
        return "—"

    if metrica in ["Nº Viagens", "Nº Paletes"]:
        return f"{valor:,.0f}".replace(",", " ")

    if metrica == "Custo/Palete (€)":
        return f"{valor:.2f} €".replace(".", ",")

    return f"{valor:.1f}".replace(".", ",")


# ==========================
# 7. Preparar tabelas
# ==========================
meses_slide = list(meses.values())
linhas_mensais = [["2026"] + meses_slide]

for metrica in metricas:
    linhas_mensais.append(
        [metrica] + [
            formatar_capilar(
                df_tabela_capilar.loc[metrica, mes],
                metrica
            )
            for mes in meses_slide
        ]
    )

linhas_comparacao = [
    ["Métrica", "Set/2026", "Out/2026", "Δ %"]
]

for _, linha in df_comparacao_capilar.iterrows():
    metrica = linha["Métrica"]
    variacao = linha["Variação %"]

    linhas_comparacao.append([
        metrica,
        formatar_capilar(linha["Setembro"], metrica),
        formatar_capilar(linha["Outubro"], metrica),
        (
            f"{variacao:+.1f}%".replace(".", ",")
            if pd.notna(variacao) else "—"
        )
    ])

# ==========================
# 8. Cabeçalho
# ==========================
texto_capilar(
    0.55, 0.38, 8, 0.5,
    "5. TRANSPORTES",
    23, AZUL, True
)

texto_capilar(
    0.55, 0.93, 9, 0.3,
    "Capilar — CODACT 013 | 2026",
    13, RGBColor(240, 95, 20), True
)

# ==========================
# 9. Tabelas PowerPoint
# ==========================
tabela_capilar(
    0.50, 1.65,
    [1.45] + [0.68] * 10,
    0.49,
    linhas_mensais,
    colunas_destaque=[9, 10]
)

tabela_capilar(
    9.00, 1.65,
    [1.50, 0.90, 0.90, 0.78],
    0.49,
    linhas_comparacao
)

# ==========================
# 10. Destaque Setembro e Outubro
# ==========================
destaque = slide.shapes.add_shape(
    MSO_SHAPE.ROUNDED_RECTANGLE,
    Inches(6.71),
    Inches(1.60),
    Inches(1.42),
    Inches(2.55)
)

destaque.fill.background()
destaque.line.color.rgb = VERMELHO
destaque.line.width = Pt(1.5)

# ==========================
# 11. Nota e rodapé
# ==========================
texto_capilar(
    9.05, 4.45, 3.75, 0.8,
    "Comparação Setembro vs Outubro de 2026.\n"
    "Universo: exclusivamente CODACT 013 (CAPILAR).",
    10, PRETO
)

texto_capilar(
    0.55, 6.95, 4.5, 0.3,
    "SALVESEN LOGÍSTICA  |  TRANSPORTES",
    11, VERDE, True
)

texto_capilar(
    8.30, 7.00, 4.40, 0.2,
    "CONFIDENTIAL AND PROPRIETARY MATERIAL",
    7, RGBColor(125, 125, 125),
    alinhamento=PP_ALIGN.RIGHT
)

# ==========================
# 12. Exportar PowerPoint
# ==========================
prs.save(str(OUTPUT))

print(f"PowerPoint criado: {OUTPUT}")


PowerPoint criado: /Users/rr/Downloads/Capilar_Setembro_Outubro_2026.pptx


In [ ]:

# ==========================
# 1. Preparar dados
# ==========================
import pandas as pd
from IPython.display import display

df = df_final.copy()

df["CODACT"] = pd.to_numeric(df["CODACT"], errors="coerce").astype("Int64")
df["ano"] = pd.to_numeric(df["ano"], errors="coerce")
df["mes"] = pd.to_numeric(df["mes"], errors="coerce")

for coluna in ["total_ingresso_c", "custo_total_c"]:
    df[coluna] = pd.to_numeric(df[coluna], errors="coerce").fillna(0)

df = df.loc[df["ano"].eq(2026) & df["mes"].isin([9, 10])].copy()

# ==========================
# 2. Agregar por cliente
# ==========================
resumo = (
    df.groupby(["CODACT", "ACTIVIDAD", "mes"], dropna=False)
    .agg(
        INGRESSO=("total_ingresso_c", "sum"),
        CUSTO=("custo_total_c", "sum")
    )
    .unstack("mes", fill_value=0)
)

resumo = resumo.reindex(
    columns=pd.MultiIndex.from_product(
        [["INGRESSO", "CUSTO"], [9, 10]]
    ),
    fill_value=0
)

resumo.columns = [
    "INGRESSO_SET", "INGRESSO_OUT",
    "CUSTO_SET", "CUSTO_OUT"
]

resumo = resumo.reset_index()

# ==========================
# 3. Calcular variações
# ==========================
resumo["RESULTADO_SET"] = resumo["INGRESSO_SET"] - resumo["CUSTO_SET"]
resumo["RESULTADO_OUT"] = resumo["INGRESSO_OUT"] - resumo["CUSTO_OUT"]

resumo["VAR_INGRESSO"] = resumo["INGRESSO_OUT"] - resumo["INGRESSO_SET"]
resumo["VAR_CUSTO"] = resumo["CUSTO_OUT"] - resumo["CUSTO_SET"]

resumo["VAR_RESULTADO"] = (
    resumo["RESULTADO_OUT"] - resumo["RESULTADO_SET"]
)

resumo["CODACT"] = resumo["CODACT"].astype("string").str.zfill(3)

# ==========================
# 4. Ordenar e apresentar
# ==========================
resumo = resumo[
    [
        "CODACT", "ACTIVIDAD",
        "INGRESSO_SET", "CUSTO_SET",
        "INGRESSO_OUT", "CUSTO_OUT",
        "VAR_INGRESSO", "VAR_CUSTO",
        "RESULTADO_SET", "RESULTADO_OUT",
        "VAR_RESULTADO"
    ]
].sort_values("VAR_RESULTADO", ascending=True).reset_index(drop=True)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)

display(
    resumo.style
    .format({
        col: "{:,.2f} €"
        for col in resumo.columns
        if col not in ["CODACT", "ACTIVIDAD"]
    })
    .map(
        lambda x: "color: red" if x < 0 else "",
        subset=["VAR_RESULTADO"]
    )
)


,CODACT,ACTIVIDAD,INGRESSO_SET,CUSTO_SET,INGRESSO_OUT,CUSTO_OUT,VAR_INGRESSO,VAR_CUSTO,RESULTADO_SET,RESULTADO_OUT,VAR_RESULTADO
0,013,DANONE PORTUGAL CAPILAR,"93,000.00 €","72,328.85 €","85,000.00 €","68,891.81 €","-8,000.00 €","-3,437.04 €","20,671.15 €","16,108.19 €","-4,562.96 €"
1,311,ALPRO PORTUGAL,"20,878.61 €","36,879.47 €","21,751.98 €","41,684.07 €",873.37 €,"4,804.60 €","-16,000.86 €","-19,932.09 €","-3,931.23 €"
2,118,ORANGNA SCHWEPPES PORTUGAL,"31,723.53 €","9,179.19 €","27,304.70 €","8,263.64 €","-4,418.83 €",-915.55 €,"22,544.34 €","19,041.06 €","-3,503.28 €"
3,074,NUTRICIA,"57,262.43 €","67,071.32 €","59,863.22 €","71,366.96 €","2,600.78 €","4,295.63 €","-9,808.89 €","-11,503.74 €","-1,694.85 €"
4,415,LOUROFOODS,"20,547.54 €","21,675.70 €","19,962.45 €","22,710.72 €",-585.09 €,"1,035.02 €","-1,128.16 €","-2,748.27 €","-1,620.11 €"
5,260,ANDROS REFRIGERADO,"27,100.18 €","19,824.03 €","27,516.31 €","21,846.16 €",416.13 €,"2,022.13 €","7,276.15 €","5,670.15 €","-1,606.00 €"
6,404,RTM,"15,377.21 €","17,395.17 €","14,945.51 €","18,224.72 €",-431.70 €,829.55 €,"-2,017.96 €","-3,279.21 €","-1,261.26 €"
7,551,UPFIELD,"19,200.00 €","19,897.79 €","19,800.00 €","21,451.60 €",600.00 €,"1,553.81 €",-697.79 €,"-1,651.60 €",-953.81 €
8,413,ARCADIE,"4,405.00 €",850.39 €,"3,612.10 €",955.55 €,-792.90 €,105.15 €,"3,554.61 €","2,656.55 €",-898.05 €
9,403,CARMONTI,"7,427.78 €","7,411.39 €","7,619.98 €","8,121.66 €",192.20 €,710.27 €,16.40 €,-501.68 €,-518.07 €


In [ ]:

# ==========================
# 1. Preparar dados
# ==========================
import pandas as pd
from IPython.display import display

df = df_final.copy()

df["ano"] = pd.to_numeric(df["ano"], errors="coerce")
df["mes"] = pd.to_numeric(df["mes"], errors="coerce")

for coluna in ["total_ingresso_c", "custo_total_c"]:
    df[coluna] = pd.to_numeric(
        df[coluna], errors="coerce"
    ).fillna(0)

df = df.loc[
    df["ano"].eq(2026)
    & df["mes"].isin([9, 10])
].copy()

# ==========================
# 2. Agregar por transportador
# ==========================
resumo_transportadores = (
    df.groupby(["TRANSPORTISTA", "mes"], dropna=False)
    .agg(
        INGRESSO=("total_ingresso_c", "sum"),
        CUSTO=("custo_total_c", "sum")
    )
    .unstack("mes", fill_value=0)
)

resumo_transportadores = resumo_transportadores.reindex(
    columns=pd.MultiIndex.from_product(
        [["INGRESSO", "CUSTO"], [9, 10]]
    ),
    fill_value=0
)

resumo_transportadores.columns = [
    "INGRESSO_SET", "INGRESSO_OUT",
    "CUSTO_SET", "CUSTO_OUT"
]

resumo_transportadores = resumo_transportadores.reset_index()

# ==========================
# 3. Calcular variações
# ==========================
resumo_transportadores["RESULTADO_SET"] = (
    resumo_transportadores["INGRESSO_SET"]
    - resumo_transportadores["CUSTO_SET"]
)

resumo_transportadores["RESULTADO_OUT"] = (
    resumo_transportadores["INGRESSO_OUT"]
    - resumo_transportadores["CUSTO_OUT"]
)

resumo_transportadores["VAR_INGRESSO"] = (
    resumo_transportadores["INGRESSO_OUT"]
    - resumo_transportadores["INGRESSO_SET"]
)

resumo_transportadores["VAR_CUSTO"] = (
    resumo_transportadores["CUSTO_OUT"]
    - resumo_transportadores["CUSTO_SET"]
)

resumo_transportadores["VAR_RESULTADO"] = (
    resumo_transportadores["RESULTADO_OUT"]
    - resumo_transportadores["RESULTADO_SET"]
)

# ==========================
# 4. Ordenar tabela
# ==========================
resumo_transportadores = resumo_transportadores[
    [
        "TRANSPORTISTA",
        "INGRESSO_SET", "CUSTO_SET",
        "INGRESSO_OUT", "CUSTO_OUT",
        "VAR_INGRESSO", "VAR_CUSTO",
        "RESULTADO_SET", "RESULTADO_OUT",
        "VAR_RESULTADO"
    ]
].sort_values(
    "VAR_RESULTADO", ascending=True
).reset_index(drop=True)

# ==========================
# 5. Mostrar tabela completa
# ==========================
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)

display(
    resumo_transportadores.style
    .format({
        coluna: "{:,.2f} €"
        for coluna in resumo_transportadores.columns
        if coluna != "TRANSPORTISTA"
    })
    .map(
        lambda valor: "color: red" if valor < 0 else "",
        subset=["VAR_RESULTADO"]
    )
)


,TRANSPORTISTA,INGRESSO_SET,CUSTO_SET,INGRESSO_OUT,CUSTO_OUT,VAR_INGRESSO,VAR_CUSTO,RESULTADO_SET,RESULTADO_OUT,VAR_RESULTADO
0,"TRANSPORTES FLORENCIO SILVA, LDA. -","207,119.96 €","225,038.17 €","206,173.96 €","232,172.68 €",-946.00 €,"7,134.51 €","-17,918.21 €","-25,998.72 €","-8,080.52 €"
1,TORRESTIR - TRANSPORTES NACIONAIS E INTERNACIONAIS S.A.,"17,876.23 €","27,436.23 €","18,961.69 €","30,482.02 €","1,085.46 €","3,045.79 €","-9,560.00 €","-11,520.33 €","-1,960.33 €"
2,MODELO CONTINENTE HIPERMERCADOS S.A.,"61,928.00 €","47,363.42 €","60,946.55 €","47,910.08 €",-981.45 €,546.65 €,"14,564.58 €","13,036.48 €","-1,528.10 €"
3,WORLD RUNNERS UNIPESSOAL LDA,"12,034.22 €","10,553.80 €",0.00 €,0.00 €,"-12,034.22 €","-10,553.80 €","1,480.42 €",0.00 €,"-1,480.42 €"
4,"TRANSPORTES FIGUEIREDO & FIGUEIREDO, LDA","10,965.33 €","12,627.52 €","17,934.33 €","20,978.31 €","6,969.00 €","8,350.79 €","-1,662.19 €","-3,043.97 €","-1,381.79 €"
5,TRANSPORTES CARGAKER LDA,"1,526.63 €","1,362.90 €","9,344.11 €","10,530.19 €","7,817.49 €","9,167.30 €",163.73 €,"-1,186.08 €","-1,349.81 €"
6,Transportes Fernando Simões Monteiro Unip. Lda,"57,233.28 €","39,736.52 €","55,887.65 €","39,379.55 €","-1,345.63 €",-356.97 €,"17,496.76 €","16,508.10 €",-988.66 €
7,JMR PRESTAÇAO DE SERVIÇOS PARA A DISTRIBUÇAO S.A,"82,037.10 €","68,134.67 €","86,161.44 €","73,062.76 €","4,124.34 €","4,928.09 €","13,902.43 €","13,098.68 €",-803.75 €
8,"Logislink - Terminal Logistica, Lda","5,616.45 €","3,851.08 €","2,221.65 €","1,240.20 €","-3,394.80 €","-2,610.88 €","1,765.37 €",981.45 €,-783.92 €
9,MOSCA PORTUGAL LDA,0.00 €,0.00 €,0.00 €,589.16 €,0.00 €,589.16 €,0.00 €,-589.16 €,-589.16 €


In [64]:

# ==========================
# 1. Bibliotecas
# ==========================
import pandas as pd
import numpy as np
from IPython.display import display

# ==========================
# 2. Preparar viagens
# ==========================
df_fill_real = df_viagens[
    ["CODEUT", "data", "paletes", "capacidade_original"]
].copy()

df_fill_real["data"] = pd.to_datetime(
    df_fill_real["data"], errors="coerce"
)

for coluna in ["paletes", "capacidade_original"]:
    df_fill_real[coluna] = pd.to_numeric(
        df_fill_real[coluna], errors="coerce"
    )

df_fill_real = df_fill_real.loc[
    df_fill_real["data"].dt.year.eq(2026)
    & df_fill_real["data"].dt.month.between(1, 10)
].copy()

df_fill_real["MES"] = df_fill_real["data"].dt.month

# ==========================
# 3. Validar capacidade real
# ==========================
df_fill_real["VALIDA"] = (
    df_fill_real["CODEUT"].notna()
    & df_fill_real["capacidade_original"].gt(0)
    & df_fill_real["paletes"].ge(0)
)

df_fill_real["OCUPACAO_PCT"] = (
    df_fill_real["paletes"]
    / df_fill_real["capacidade_original"].replace(0, np.nan)
    * 100
)

df_fill_real["ACIMA_100"] = (
    df_fill_real["VALIDA"]
    & df_fill_real["OCUPACAO_PCT"].gt(100)
)

# ==========================
# 4. Agregar por mês
# ==========================
df_validas = df_fill_real.loc[
    df_fill_real["VALIDA"]
].copy()

df_truck_fill_real = (
    df_validas
    .groupby("MES")
    .agg(
        VIAGENS=("CODEUT", "nunique"),
        PALETES=("paletes", "sum"),
        CAPACIDADE=("capacidade_original", "sum"),
        VIAGENS_ACIMA_100=("ACIMA_100", "sum")
    )
    .reindex(range(1, 11), fill_value=0)
)

df_truck_fill_real["TRUCK_FILL_RATE"] = (
    df_truck_fill_real["PALETES"]
    / df_truck_fill_real["CAPACIDADE"].replace(0, np.nan)
    * 100
)

# ==========================
# 5. Calcular médias
# ==========================
def media_ponderada(inicio, fim):
    dados = df_truck_fill_real.loc[inicio:fim]
    capacidade = dados["CAPACIDADE"].sum()

    if capacidade <= 0:
        return np.nan

    return dados["PALETES"].sum() / capacidade * 100

media_jan_jun = media_ponderada(1, 6)
media_jan_out = media_ponderada(1, 10)

# ==========================
# 6. Preparar tabela
# ==========================
meses = {
    1: "Jan", 2: "Fev", 3: "Mar",
    4: "Abr", 5: "Mai", 6: "Jun",
    7: "Jul", 8: "Ago", 9: "Set",
    10: "Out"
}

df_resultado_fill_real = df_truck_fill_real.reset_index()

df_resultado_fill_real["MES"] = (
    df_resultado_fill_real["MES"].map(meses)
)

df_medias = pd.DataFrame({
    "MES": ["Média Jan–Jun", "Média Jan–Out"],
    "VIAGENS": [
        df_truck_fill_real.loc[1:6, "VIAGENS"].sum(),
        df_truck_fill_real["VIAGENS"].sum()
    ],
    "PALETES": [
        df_truck_fill_real.loc[1:6, "PALETES"].sum(),
        df_truck_fill_real["PALETES"].sum()
    ],
    "CAPACIDADE": [
        df_truck_fill_real.loc[1:6, "CAPACIDADE"].sum(),
        df_truck_fill_real["CAPACIDADE"].sum()
    ],
    "VIAGENS_ACIMA_100": [
        df_truck_fill_real.loc[1:6, "VIAGENS_ACIMA_100"].sum(),
        df_truck_fill_real["VIAGENS_ACIMA_100"].sum()
    ],
    "TRUCK_FILL_RATE": [
        media_jan_jun,
        media_jan_out
    ]
})

df_resultado_fill_real = pd.concat(
    [df_resultado_fill_real, df_medias],
    ignore_index=True
)

df_resultado_fill_real = df_resultado_fill_real.rename(
    columns={"TRUCK_FILL_RATE": "TRUCK_FILL_RATE_%"}
)

# ==========================
# 7. Resultado
# ==========================
df_resultado_fill_real = df_resultado_fill_real.round(2)

display(df_resultado_fill_real)

print(
    "Viagens sem capacidade ou paletes válidas:",
    (~df_fill_real["VALIDA"]).sum()
)

print(
    "Viagens com ocupação superior a 100%:",
    df_fill_real["ACIMA_100"].sum()
)


,MES,VIAGENS,PALETES,CAPACIDADE,VIAGENS_ACIMA_100,TRUCK_FILL_RATE_%
0,Jan,1922,46297.0,54433,453,85.05
1,Fev,1718,38238.75,45489,389,84.06
2,Mar,1965,46474.56,52900,474,87.85
3,Abr,1972,48077.75,53650,477,89.61
4,Mai,1953,47682.05,52742,510,90.41
5,Jun,1966,45447.16,52951,464,85.83
6,Jul,2164,49275.5,57697,506,85.4
7,Ago,1442,29781.0,33491,351,88.92
8,Set,1418,29209.5,33792,403,86.44
9,Out,2095,49841.18,53715,679,92.79


Viagens sem capacidade ou paletes válidas: 1254
Viagens com ocupação superior a 100%: 4706


In [66]:

# ==========================
# 1. Bibliotecas
# ==========================
import pandas as pd
import numpy as np
from IPython.display import display

# ==========================
# 2. Preparar viagens
# ==========================
df_fill_real = df_viagens[
    ["CODEUT", "data", "paletes", "capacidade_original"]
].copy()

df_fill_real["data"] = pd.to_datetime(
    df_fill_real["data"], errors="coerce"
)

for coluna in ["paletes", "capacidade_original"]:
    df_fill_real[coluna] = pd.to_numeric(
        df_fill_real[coluna], errors="coerce"
    )

df_fill_real = df_fill_real.loc[
    df_fill_real["data"].dt.year.eq(2026)
    & df_fill_real["data"].dt.month.between(1, 10)
].copy()

df_fill_real["MES"] = df_fill_real["data"].dt.month

# ==========================
# 3. Capacidade duplo deck
# ==========================
capacidade = df_fill_real["capacidade_original"]
paletes = df_fill_real["paletes"]

# Capacidade máxima de 66 paletes
df_fill_real["CAPACIDADE_UTILIZADA"] = capacidade.clip(upper=66)

# Veículos de 33 com mais de 45 paletes: duplo deck de 55
df_fill_real.loc[
    capacidade.eq(33) & paletes.gt(45),
    "CAPACIDADE_UTILIZADA"
] = 55

# ==========================
# 4. Validar ocupação
# ==========================
df_fill_real["VALIDA"] = (
    df_fill_real["CODEUT"].notna()
    & df_fill_real["CAPACIDADE_UTILIZADA"].gt(0)
    & paletes.ge(0)
)

df_fill_real["OCUPACAO_PCT"] = (
    df_fill_real["paletes"]
    / df_fill_real["CAPACIDADE_UTILIZADA"].replace(0, np.nan)
    * 100
)

df_fill_real["ACIMA_100"] = (
    df_fill_real["VALIDA"]
    & df_fill_real["OCUPACAO_PCT"].gt(100)
)

# ==========================
# 5. Agregar por mês
# ==========================
df_validas = df_fill_real.loc[
    df_fill_real["VALIDA"]
].copy()

df_truck_fill_real = (
    df_validas
    .groupby("MES")
    .agg(
        VIAGENS=("CODEUT", "nunique"),
        PALETES=("paletes", "sum"),
        CAPACIDADE=("CAPACIDADE_UTILIZADA", "sum"),
        VIAGENS_ACIMA_100=("ACIMA_100", "sum")
    )
    .reindex(range(1, 11), fill_value=0)
)

df_truck_fill_real["TRUCK_FILL_RATE"] = (
    df_truck_fill_real["PALETES"]
    / df_truck_fill_real["CAPACIDADE"].replace(0, np.nan)
    * 100
)

# ==========================
# 6. Médias ponderadas
# ==========================
def media_ponderada(inicio, fim):
    dados = df_truck_fill_real.loc[inicio:fim]

    capacidade_total = dados["CAPACIDADE"].sum()

    if capacidade_total <= 0:
        return np.nan

    return dados["PALETES"].sum() / capacidade_total * 100

media_jan_jun = media_ponderada(1, 6)
media_jan_out = media_ponderada(1, 10)

# ==========================
# 7. Preparar tabela
# ==========================
meses = {
    1: "Jan",
    2: "Fev",
    3: "Mar",
    4: "Abr",
    5: "Mai",
    6: "Jun",
    7: "Jul",
    8: "Ago",
    9: "Set",
    10: "Out"
}

df_resultado_fill_real = df_truck_fill_real.reset_index()

df_resultado_fill_real["MES"] = (
    df_resultado_fill_real["MES"].map(meses)
)

df_medias = pd.DataFrame({
    "MES": ["Média Jan–Jun", "Média Jan–Out"],
    "VIAGENS": [
        df_truck_fill_real.loc[1:6, "VIAGENS"].sum(),
        df_truck_fill_real["VIAGENS"].sum()
    ],
    "PALETES": [
        df_truck_fill_real.loc[1:6, "PALETES"].sum(),
        df_truck_fill_real["PALETES"].sum()
    ],
    "CAPACIDADE": [
        df_truck_fill_real.loc[1:6, "CAPACIDADE"].sum(),
        df_truck_fill_real["CAPACIDADE"].sum()
    ],
    "VIAGENS_ACIMA_100": [
        df_truck_fill_real.loc[1:6, "VIAGENS_ACIMA_100"].sum(),
        df_truck_fill_real["VIAGENS_ACIMA_100"].sum()
    ],
    "TRUCK_FILL_RATE": [
        media_jan_jun,
        media_jan_out
    ]
})

df_resultado_fill_real = pd.concat(
    [df_resultado_fill_real, df_medias],
    ignore_index=True
)

df_resultado_fill_real = df_resultado_fill_real.rename(
    columns={"TRUCK_FILL_RATE": "TRUCK_FILL_RATE_%"}
)

# ==========================
# 8. Controlo de qualidade
# ==========================
df_controlo_fill = pd.DataFrame({
    "INDICADOR": [
        "Viagens analisadas",
        "Viagens válidas",
        "Viagens sem dados válidos",
        "Viagens 33 convertidas para 55",
        "Viagens com ocupação superior a 100%"
    ],
    "VALOR": [
        len(df_fill_real),
        len(df_validas),
        (~df_fill_real["VALIDA"]).sum(),
        (
            capacidade.eq(33)
            & paletes.gt(45)
        ).sum(),
        df_fill_real["ACIMA_100"].sum()
    ]
})

# ==========================
# 9. Resultados
# ==========================
display(df_resultado_fill_real.round(2))
display(df_controlo_fill)


,MES,VIAGENS,PALETES,CAPACIDADE,VIAGENS_ACIMA_100,TRUCK_FILL_RATE_%
0,Jan,1922,46297.0,58129,381,79.65
1,Fev,1718,38238.75,48129,340,79.45
2,Mar,1965,46474.56,56244,417,82.63
3,Abr,1972,48077.75,57038,424,84.29
4,Mai,1953,47682.05,56240,455,84.78
5,Jun,1966,45447.16,55789,426,81.46
6,Jul,2164,49275.5,60887,442,80.93
7,Ago,1442,29781.0,35053,335,84.96
8,Set,1418,29209.5,35310,382,82.72
9,Out,2095,49841.18,57411,602,86.81


,INDICADOR,VALOR
0,Viagens analisadas,19869
1,Viagens válidas,18615
2,Viagens sem dados válidos,1254
3,Viagens 33 convertidas para 55,1335
4,Viagens com ocupação superior a 100%,4204


In [67]:

# ==========================
# 1. Mês e Truck Fill Rate
# ==========================
df_fill_mensal = df_resultado_fill_real.loc[
    ~df_resultado_fill_real["MES"].str.startswith("Média"),
    ["MES", "TRUCK_FILL_RATE_%"]
].copy()

df_fill_mensal = df_fill_mensal.rename(
    columns={"TRUCK_FILL_RATE_%": "Truck Fill Rate (%)"}
)

# ==========================
# 2. Resultado
# ==========================
df_fill_mensal.round(2)


,MES,Truck Fill Rate (%)
0,Jan,79.65
1,Fev,79.45
2,Mar,82.63
3,Abr,84.29
4,Mai,84.78
5,Jun,81.46
6,Jul,80.93
7,Ago,84.96
8,Set,82.72
9,Out,86.81


In [ ]:

# ==========================
# 1. Bibliotecas
# ==========================
import duckdb
import pandas as pd
import numpy as np

# ==========================
# 2. Consultar CODACT 013
# ==========================
with duckdb.connect(str(PATHS["db"]), read_only=True) as con:
    df_fill_013 = con.execute("""
        WITH origem AS (
            SELECT
                SUBSTR(TRIM(CAST(FENTREGA AS VARCHAR)), 1, 6) AS MES,
                NULLIF(TRIM(CAST(CODEUT AS VARCHAR)), '') AS CODEUT,
                TRY_CAST(
                    REPLACE(TRIM(CAST(PALETS AS VARCHAR)), ',', '.')
                    AS DOUBLE
                ) AS PALETES,
                TRY_CAST(
                    REPLACE(
                        TRIM(CAST(CAMION_CAPACIDAD AS VARCHAR)),
                        ',', '.'
                    ) AS DOUBLE
                ) AS CAPACIDADE
            FROM inform_27_2026
            WHERE TRY_CAST(CODACT AS INTEGER) != 13
              AND SUBSTR(TRIM(CAST(FENTREGA AS VARCHAR)), 1, 6)
                  BETWEEN '202601' AND '202610'
        )
        SELECT
            MES,
            CODEUT,
            SUM(PALETES) AS PALETES,
            MAX(CAPACIDADE) AS CAPACIDADE_ORIGINAL
        FROM origem
        WHERE CODEUT IS NOT NULL
        GROUP BY MES, CODEUT
    """).df()

# ==========================
# 3. Capacidade duplo deck
# ==========================
df_fill_013["CAPACIDADE_UTILIZADA"] = (
    df_fill_013["CAPACIDADE_ORIGINAL"].clip(upper=66)
)

df_fill_013.loc[
    df_fill_013["CAPACIDADE_ORIGINAL"].eq(33)
    & df_fill_013["PALETES"].gt(45),
    "CAPACIDADE_UTILIZADA"
] = 55

# ==========================
# 4. Validar viagens
# ==========================
df_fill_013["VALIDA"] = (
    df_fill_013["PALETES"].ge(0)
    & df_fill_013["CAPACIDADE_UTILIZADA"].gt(0)
)

df_fill_013_validas = df_fill_013.loc[
    df_fill_013["VALIDA"]
].copy()

# ==========================
# 5. Calcular ocupação mensal
# ==========================
df_resultado_013 = (
    df_fill_013_validas
    .groupby("MES")
    .agg(
        PALETES=("PALETES", "sum"),
        CAPACIDADE=("CAPACIDADE_UTILIZADA", "sum")
    )
    .reindex(
        [f"2026{m:02d}" for m in range(1, 11)]
    )
)

df_resultado_013["Truck Fill Rate (%)"] = (
    df_resultado_013["PALETES"]
    / df_resultado_013["CAPACIDADE"].replace(0, np.nan)
    * 100
)

# ==========================
# 6. Resultado
# ==========================
meses = {
    f"2026{m:02d}": nome
    for m, nome in enumerate(
        ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun",
         "Jul", "Ago", "Set", "Out"],
        start=1
    )
}

df_fill_mensal_013 = (
    df_resultado_013
    .reset_index()
    .rename(columns={"MES": "Mês"})
)

df_fill_mensal_013["Mês"] = (
    df_fill_mensal_013["Mês"].map(meses)
)

df_fill_mensal_013 = df_fill_mensal_013[
    ["Mês", "Truck Fill Rate (%)"]
].round(2)

df_fill_mensal_013


,Mês,Truck Fill Rate (%)
0,Jan,78.14
1,Fev,84.00
2,Mar,86.63
3,Abr,87.73
4,Mai,95.22
5,Jun,65.42
6,Jul,53.93
7,Ago,79.06
8,Set,70.54
9,Out,85.64
